# Play Santorini against the agent

This notebook loads your trained network and opens a board where you play against the agent and watch it think.

**The agent** is the trained network plus the same MCTS used in self-play, with the exact df-pn solver on the root. Each move it runs the number of simulations you choose: 64 (quick) up to 4,000 (deep).

**What the widget shows of the agent's thinking:**
- **Arrows on the board** for the moves it is weighing; thicker means more simulations. Squares mark where it would build.
- **"Which moves did it try?"**: simulations per move as bars, the network's prior (its guess before searching) as dashed outlines, and the win chance after search on top. Hover a bar to draw its expected line on the board.
- **The search tree**: every box is a position the search reached, with a mini-board, visit count N, the win chance for the player who just moved, the network's first impression and prior, and optionally the PUCT score (Q + U). The most-visited line is outlined. Click a box to explore from there; hover it to see the position on the big board. Prune it with max nodes, children kept and max depth; zoom, drag and full screen.
- **Step through the search**: Select (walk down by highest Q + U), Expand and evaluate (one network call), Back up (update the path). Each step is explained in words and highlighted in the tree. Tick "Pause the agent on its turn" to step through the agent's own search before it moves.
- **Game timeline**: drag the slider, or click a move or a point on the chart, to review any earlier move with the thinking, bars and tree behind it. "Play from here" continues the game from one of your earlier turns.
- **Winning chances over the game**, filled toward whichever side the agent thinks is ahead.

When the agent places first, the board shades each cell by how good it is for the agent's workers. Turn on **Hints on my turn** to get the same analysis for your own moves.

**Runtime:** any. CPU is fine up to "Normal"; a GPU makes "Strong" and "Deep" fast. Run all cells, then play in the board under the last cell.

In [12]:
import os, sys, subprocess, sysconfig, time
import importlib.metadata as md
T0 = time.time()
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')} | {(time.time() - T0) / 60:5.1f} min] {msg}", flush=True)

# Same package fix as the training notebooks: upgrade Flax/optax, keep the installed JAX.
installed = {d.metadata["Name"].lower() for d in md.distributions()}
flax_before = md.version("flax") if "flax" in installed else None
pins = [f"{p}=={md.version(p)}" for p in ("jax", "jaxlib", "libtpu", "libtpu-nightly", "jax-cuda12-plugin",
        "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt") if p in installed]
open("constraints.txt", "w").write("\n".join(pins) + "\n")
log("installing pybind11, upgrading flax/optax ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pybind11", "flax", "optax", "-c", "constraints.txt"], check=True)
if "flax" in sys.modules and flax_before != md.version("flax"):
    raise SystemExit("Flax was upgraded after it was imported: Runtime -> Restart session, then run all again.")
import jax
log(f"JAX {jax.__version__}, Flax {md.version('flax')}, backend {jax.default_backend()} ({jax.devices()[0].device_kind})")

[15:55:19 |   0.0 min] installing pybind11, upgrading flax/optax ...
[15:55:24 |   0.1 min] JAX 0.11.1, Flax 0.12.12, backend cpu (cpu)


## Source files
The engine, solver, search, agent and widget, written to the runtime.

In [13]:
%%writefile engine.h
// Santorini (base game, no god powers) engine: bitboards, move generation,
// forced-move pruning, D4 symmetry and canonical Zobrist keys.
//
// Board: 25 cells, index = row*5 + col, row 0 = top (row "1"), col 0 = "A".
// Positions are stored from the side-to-move's perspective: `me` moves next.
#pragma once
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <random>
#include <algorithm>

typedef uint32_t BB;
constexpr BB FULL = (1u << 25) - 1;
inline int lsb(BB b) { return __builtin_ctz(b); }
inline int pc(BB b) { return __builtin_popcount(b); }
inline BB bit(int c) { return 1u << c; }

// Official piece supply: 22 level-1, 18 level-2, 14 level-3 blocks, 18 domes.
constexpr int LIMIT1 = 22, LIMIT2 = 18, LIMIT3 = 14, LIMITD = 18;

struct Tables {
    BB adj[25];
    uint8_t sym[8][25];          // D4 symmetry: sym[s][cell] -> image cell
    uint64_t zh[25][5];          // height keys (zh[c][0] = 0)
    uint64_t zme[25], zop[25];   // worker keys (mover / opponent)
};
inline Tables T;

inline void initTables() {
    for (int c = 0; c < 25; c++) {
        int r = c / 5, k = c % 5;
        BB m = 0;
        for (int dr = -1; dr <= 1; dr++)
            for (int dk = -1; dk <= 1; dk++) {
                if (!dr && !dk) continue;
                int rr = r + dr, kk = k + dk;
                if (rr >= 0 && rr < 5 && kk >= 0 && kk < 5) m |= bit(rr * 5 + kk);
            }
        T.adj[c] = m;
        for (int s = 0; s < 8; s++) {
            int x = k, y = r;              // rotate s&3 times, then mirror if s>=4
            for (int i = 0; i < (s & 3); i++) { int nx = 4 - y, ny = x; x = nx; y = ny; }
            if (s >= 4) x = 4 - x;
            T.sym[s][c] = (uint8_t)(y * 5 + x);
        }
    }
    std::mt19937_64 rng(0x5a170421ULL);
    for (int c = 0; c < 25; c++) {
        T.zh[c][0] = 0;
        for (int h = 1; h < 5; h++) T.zh[c][h] = rng();
        T.zme[c] = rng();
        T.zop[c] = rng();
    }
}

struct Pos {
    BB h1 = 0, h2 = 0, h3 = 0, dm = 0;  // height >=1, >=2, >=3, dome
    BB me = 0, op = 0;                  // worker sets

    int height(int c) const {
        return ((h1 >> c) & 1) + ((h2 >> c) & 1) + ((h3 >> c) & 1) + ((dm >> c) & 1);
    }
    BB lvl(int l) const {
        switch (l) {
            case 0: return FULL & ~h1;
            case 1: return h1 & ~h2;
            case 2: return h2 & ~h3;
            case 3: return h3 & ~dm;
            default: return dm;
        }
    }
    int blocks() const { return pc(h1) + pc(h2) + pc(h3) + pc(dm); }
    void raise(int c) {
        BB b = bit(c);
        if (!(h1 & b)) h1 |= b;
        else if (!(h2 & b)) h2 |= b;
        else if (!(h3 & b)) h3 |= b;
        else dm |= b;
    }
    // Squares on which a block can currently be placed (supply limits respected).
    BB buildable() const {
        BB m = 0;
        if (pc(h1) < LIMIT1) m |= lvl(0);
        if (pc(h2) < LIMIT2) m |= lvl(1);
        if (pc(h3) < LIMIT3) m |= lvl(2);
        if (pc(dm) < LIMITD) m |= lvl(3);
        return m;
    }
};

struct Move { uint8_t f, t, b; };

// Squares a worker standing at height `lw` may step to (ignoring occupancy).
inline BB climbMask(const Pos& p, int lw) {
    if (lw == 0) return ~p.h2 & ~p.dm & FULL;
    if (lw == 1) return ~p.h3 & ~p.dm & FULL;
    return ~p.dm & FULL;
}

// Can the side owning `w` win right now by stepping from level 2 onto level 3?
inline bool canWinNow(const Pos& p, BB w) {
    BB occ = p.me | p.op, L2 = p.lvl(2), L3 = p.lvl(3);
    for (BB x = w & L2; x; x &= x - 1)
        if (T.adj[lsb(x)] & L3 & ~occ) return true;
    return false;
}

// Result of move generation with forced-move pruning.
//   win  : mover can win immediately (no need to look at moves)
//   n == 0 (and !win): mover is lost (no legal move, or every move hands the
//          opponent an immediate win)
// Otherwise m[0..n) are exactly the moves that do NOT let the opponent win
// on the very next step. Dropping the others is value-preserving: such a move
// loses, so the mover only needs them if nothing else exists (then: loss).
struct Gen {
    bool win;
    int n;
    Move m[128];
};

inline void genSafe(const Pos& p, Gen& g) {
    g.win = false;
    g.n = 0;
    if (canWinNow(p, p.me)) { g.win = true; return; }
    BB occ = p.me | p.op;
    BB L2 = p.lvl(2), L3 = p.lvl(3);
    BB oppAdj = 0;  // squares adjacent to an opponent worker standing on level 2
    for (BB x = p.op & L2; x; x &= x - 1) oppAdj |= T.adj[lsb(x)];
    BB danger2 = L2 & oppAdj;       // raising these to level 3 hands over a win
    BB bmAllowed = p.buildable();
    for (BB ws = p.me; ws; ws &= ws - 1) {
        int w = lsb(ws);
        BB dests = T.adj[w] & ~occ & climbMask(p, p.height(w));
        for (; dests; dests &= dests - 1) {
            int t = lsb(dests);
            BB occ2 = occ ^ bit(w) ^ bit(t);
            BB bm = T.adj[t] & ~occ2 & bmAllowed;
            BB threats = oppAdj & L3 & ~occ2;   // open level-3 squares the opponent can step onto
            BB safe;
            if (!threats) safe = bm & ~danger2;
            else if (!(threats & (threats - 1))) safe = bm & threats;  // must dome the single threat
            else safe = 0;                                              // two threats: cannot stop both
            for (; safe; safe &= safe - 1)
                g.m[g.n++] = Move{(uint8_t)w, (uint8_t)t, (uint8_t)lsb(safe)};
        }
    }
}

// Apply a move and flip perspective (opponent becomes the mover).
inline Pos makeChild(const Pos& p, Move m) {
    Pos c = p;
    c.raise(m.b);
    BB me2 = p.me ^ bit(m.f) ^ bit(m.t);
    c.me = p.op;
    c.op = me2;
    return c;
}

// ---- Symmetry-aware hashing ------------------------------------------------
// A Node carries the height hash under each of the 8 symmetries so a child's
// canonical key costs ~16 table lookups instead of re-hashing the board.
struct Node {
    Pos p;
    uint64_t hb[8];
};

inline Node makeRoot(const Pos& p) {
    Node n;
    n.p = p;
    for (int s = 0; s < 8; s++) {
        uint64_t h = 0;
        for (int c = 0; c < 25; c++) h ^= T.zh[T.sym[s][c]][p.height(c)];
        n.hb[s] = h;
    }
    return n;
}

inline uint64_t workersKey(int s, BB me, BB op) {
    uint64_t k = 0;
    for (BB x = me; x; x &= x - 1) k ^= T.zme[T.sym[s][lsb(x)]];
    for (BB x = op; x; x &= x - 1) k ^= T.zop[T.sym[s][lsb(x)]];
    return k;
}

inline uint64_t canonicalKey(const Node& n) {
    uint64_t best = ~0ULL;
    for (int s = 0; s < 8; s++) best = std::min(best, n.hb[s] ^ workersKey(s, n.p.me, n.p.op));
    return best;
}

inline Node childNode(const Node& n, Move m) {
    Node c;
    c.p = makeChild(n.p, m);
    int h0 = n.p.height(m.b);
    for (int s = 0; s < 8; s++) {
        int img = T.sym[s][m.b];
        c.hb[s] = n.hb[s] ^ T.zh[img][h0] ^ T.zh[img][h0 + 1];
    }
    return c;
}

// Fast canonical key of a child without building the full Node.
struct KeyCtx {
    uint64_t hb[8], meAsOp[8];  // child's opponent = parent's mover (moved worker added per move)
    uint64_t opAsMe[8];         // child's mover = parent's opponent (unchanged)
    void init(const Node& n) {
        for (int s = 0; s < 8; s++) {
            hb[s] = n.hb[s];
            opAsMe[s] = 0;
            for (BB x = n.p.op; x; x &= x - 1) opAsMe[s] ^= T.zme[T.sym[s][lsb(x)]];
        }
    }
    uint64_t childKey(const Node& n, Move m) const {
        int h0 = n.p.height(m.b);
        int other = lsb(n.p.me ^ bit(m.f));
        uint64_t best = ~0ULL;
        for (int s = 0; s < 8; s++) {
            const uint8_t* S = T.sym[s];
            uint64_t k = hb[s] ^ T.zh[S[m.b]][h0] ^ T.zh[S[m.b]][h0 + 1] ^ opAsMe[s] ^
                         T.zop[S[m.t]] ^ T.zop[S[other]];
            best = std::min(best, k);
        }
        return best;
    }
};

// ---- I/O helpers -------------------------------------------------------------
inline std::string sq(int c) {
    std::string s;
    s += char('A' + c % 5);
    s += char('1' + c / 5);
    return s;
}
inline int parseSq(const std::string& s) {
    return (s[1] - '1') * 5 + (toupper(s[0]) - 'A');
}
inline std::string moveStr(Move m) { return sq(m.f) + "-" + sq(m.t) + " build " + sq(m.b); }

// heights: 25 digits, row 1 (top) first. me/op: "D3,C5".
inline Pos parsePos(const std::string& heights, const std::string& me, const std::string& op) {
    Pos p;
    for (int c = 0; c < 25; c++)
        for (int i = 0; i < heights[c] - '0'; i++) p.raise(c);
    auto workers = [](const std::string& s) {
        BB b = 0;
        b |= bit(parseSq(s.substr(0, 2)));
        b |= bit(parseSq(s.substr(3, 2)));
        return b;
    };
    p.me = workers(me);
    p.op = workers(op);
    return p;
}

inline void printPos(const Pos& p, const char* meName = "X", const char* opName = "O") {
    printf("     A   B   C   D   E\n");
    for (int r = 0; r < 5; r++) {
        printf("  %d ", r + 1);
        for (int k = 0; k < 5; k++) {
            int c = r * 5 + k, h = p.height(c);
            char w = (p.me >> c & 1) ? meName[0] : (p.op >> c & 1) ? opName[0] : ' ';
            if (h == 4) printf(" [] ");
            else printf(" %d%c ", h, w);
        }
        printf("\n");
    }
    printf("  (%c = side to move, %c = opponent, [] = dome)\n", meName[0], opName[0]);
}

Overwriting engine.h


In [14]:
%%writefile solver.h
// Transposition table + depth-first proof-number search (df-pn) with the
// 1+epsilon threshold trick. Values are from the side-to-move's perspective:
//   pn == 0  -> mover wins (proven)      dn == 0 -> mover loses (proven)
// Santorini's game graph is a DAG (every turn adds a block), so there is no
// repetition / graph-history-interaction problem for df-pn.
#pragma once
#include "engine.h"
#include <atomic>
#include <chrono>
#include <cmath>
#include <vector>

constexpr uint32_t INF = 0x3FFFFFFF;
inline uint32_t sat(uint64_t x) { return x >= INF ? INF : (uint32_t)x; }

// ---- Transposition table -------------------------------------------------------
// 4-way buckets of 16-byte entries (one cache line). The 48-bit tag plus the
// bucket index together verify all 64 key bits. Replacement keeps the entries
// that cost the most work to compute; proven results get a large bonus.
struct TT {
    struct Entry {
        uint64_t tagWork;  // high 48 bits: key tag, low 16 bits: log2(work) * 1024
        uint32_t pn, dn;
    };
    Entry* t = nullptr;
    uint64_t nBuckets = 0, mask = 0;
    uint64_t stores = 0, used = 0;

    TT() = default;
    TT(const TT&) = delete;
    TT& operator=(const TT&) = delete;
    ~TT() { free(t); }

    void alloc(double gb) {
        free(t);
        t = nullptr;
        uint64_t bytes = (uint64_t)(gb * (1ULL << 30));
        nBuckets = 1;
        while (nBuckets * 2 * 64 <= bytes) nBuckets *= 2;
        mask = nBuckets - 1;
        t = (Entry*)aligned_alloc(64, nBuckets * 64);
        memset(t, 0, nBuckets * 64);
    }
    static uint64_t tagOf(uint64_t key) { return key & ~0xFFFFULL; }
    Entry* bucket(uint64_t key) { return t + (key & mask) * 4; }

    bool lookup(uint64_t key, uint32_t& pn, uint32_t& dn) {
        Entry* b = bucket(key);
        uint64_t tag = tagOf(key);
        for (int i = 0; i < 4; i++)
            if ((b[i].tagWork & ~0xFFFFULL) == tag && (b[i].pn | b[i].dn)) {
                pn = b[i].pn;
                dn = b[i].dn;
                return true;
            }
        return false;
    }
    static uint32_t prio(const Entry& e) {
        uint32_t w = e.tagWork & 0xFFFF;
        if (e.pn == 0 || e.dn == 0) w += 40000;  // keep proofs
        return w;
    }
    void store(uint64_t key, uint32_t pn, uint32_t dn, uint64_t work) {
        stores++;
        Entry* b = bucket(key);
        uint64_t tag = tagOf(key);
        uint32_t w16 = std::min<uint32_t>(0xFFFF, (uint32_t)(std::log2((double)work + 1) * 1024));
        Entry* victim = nullptr;
        for (int i = 0; i < 4; i++)
            if ((b[i].tagWork & ~0xFFFFULL) == tag && (b[i].pn | b[i].dn)) { victim = &b[i]; break; }
        if (!victim) {
            for (int i = 0; i < 4; i++)
                if (!(b[i].pn | b[i].dn)) { victim = &b[i]; used++; break; }
        }
        if (!victim) {
            victim = &b[0];
            for (int i = 1; i < 4; i++)
                if (prio(b[i]) < prio(*victim)) victim = &b[i];
        }
        victim->tagWork = tag | w16;
        victim->pn = pn;
        victim->dn = dn;
    }
    double fill() const { return (double)used / (nBuckets * 4); }
};

// ---- df-pn -------------------------------------------------------------------
struct Solver {
    TT tt;
    double eps = 0.25;
    uint64_t expansions = 0;
    uint64_t nextReport = 1ULL << 22;
    std::chrono::steady_clock::time_point start, deadline;
    bool aborted = false;
    bool verbose = true;
    uint64_t budget = 0;       // max expansions per solve() call; 0 = unlimited
    uint64_t budgetStart = 0;

    double elapsed() const {
        return std::chrono::duration<double>(std::chrono::steady_clock::now() - start).count();
    }

    // Initial (pn, dn) for an unexplored child. Plain 1/1; children whose mover
    // is under an immediate threat get a smaller disproof estimate because
    // their reply is forced.
    int heuristic = 1;
    void initChild(const Pos& c, uint32_t& pn, uint32_t& dn) {
        pn = 1;
        dn = 1;
        if (!heuristic) return;
        // Threats the child's mover must answer (open level-3 squares next to an
        // opposing worker on level 2).
        BB occ = c.me | c.op, L2 = c.lvl(2), L3 = c.lvl(3);
        BB adj = 0;
        for (BB x = c.op & L2; x; x &= x - 1) adj |= T.adj[lsb(x)];
        int threats = pc(adj & L3 & ~occ);
        if (threats >= 2) { pn = 4; dn = 1; return; }
        if (threats == 1) { pn = 2; dn = 1; return; }
        int d = 0;
        for (BB x = c.me; x; x &= x - 1) d += c.height(lsb(x));
        for (BB x = c.op; x; x &= x - 1) d -= c.height(lsb(x));
        if (d > 0) { pn = 1; dn = 1 + std::min(d, 3); }
        else if (d < 0) { pn = 1 + std::min(-d, 3); dn = 1; }
    }

    uint64_t mid(const Node& n, uint64_t key, uint32_t thpn, uint32_t thdn, uint32_t& outPn,
                 uint32_t& outDn) {
        expansions++;
        if (budget && expansions - budgetStart >= budget) aborted = true;  // per-position cap
        if (expansions >= nextReport) {
            nextReport += 1ULL << 22;
            if (verbose)
                fprintf(stderr, "  [%.0fs] %llu expansions, %.1f M/s, TT fill %.1f%%\n", elapsed(),
                        (unsigned long long)expansions, expansions / elapsed() / 1e6,
                        100 * tt.fill());
            if (std::chrono::steady_clock::now() > deadline) aborted = true;
        }
        Gen g;
        genSafe(n.p, g);
        if (g.win) { outPn = 0; outDn = INF; tt.store(key, 0, INF, 1); return 1; }
        if (g.n == 0) { outPn = INF; outDn = 0; tt.store(key, INF, 0, 1); return 1; }

        KeyCtx kc;
        kc.init(n);
        uint64_t ck[128];
        uint32_t cpn[128], cdn[128];
        for (int i = 0; i < g.n; i++) {
            ck[i] = kc.childKey(n, g.m[i]);
            if (!tt.lookup(ck[i], cpn[i], cdn[i])) initChild(makeChild(n.p, g.m[i]), cpn[i], cdn[i]);
        }
        uint64_t work = 1;
        uint32_t pn, dn;
        for (;;) {
            // pn(node) = min over children of dn(child); dn(node) = sum of pn(child)
            uint32_t best = INF + 1, second = INF;
            int bi = -1;
            uint64_t sum = 0;
            for (int i = 0; i < g.n; i++) {
                sum += cpn[i];
                if (cdn[i] < best) { second = best; best = cdn[i]; bi = i; }
                else if (cdn[i] < second) second = cdn[i];
            }
            pn = best;
            dn = sat(sum);
            if (second > INF) second = INF;
            if (pn >= thpn || dn >= thdn || aborted) break;
            uint32_t cthpn = sat((uint64_t)thdn - dn + cpn[bi]);
            uint64_t widened = std::max<uint64_t>((uint64_t)second + 1,
                                                  (uint64_t)std::ceil(second * (1.0 + eps)));
            uint32_t cthdn = (uint32_t)std::min<uint64_t>(thpn, widened);
            Node c = childNode(n, g.m[bi]);
            work += mid(c, ck[bi], cthpn, cthdn, cpn[bi], cdn[bi]);
        }
        tt.store(key, pn, dn, work);
        outPn = pn;
        outDn = dn;
        return work;
    }

    // Returns +1 mover wins, -1 mover loses, 0 unresolved (time limit).
    int solve(const Pos& p, double seconds, uint32_t* pnOut = nullptr, uint32_t* dnOut = nullptr) {
        start = std::chrono::steady_clock::now();
        deadline = start + std::chrono::milliseconds((int64_t)(seconds * 1000));
        aborted = false;
        budgetStart = expansions;
        Node root = makeRoot(p);
        uint64_t key = canonicalKey(root);
        uint32_t pn, dn;
        mid(root, key, INF, INF, pn, dn);
        if (pnOut) *pnOut = pn;
        if (dnOut) *dnOut = dn;
        return pn == 0 ? 1 : dn == 0 ? -1 : 0;
    }

    // Training label for one position.
    //   value: +1 mover wins, -1 mover loses, 0 unresolved within rootBudget expansions.
    //   winning: when value == +1, every move proven to win. An immediate step-up
    //   win has no build; it is encoded with b = f (build back on the vacated cell),
    //   so it maps to a fixed (worker, move dir, build dir) action index.
    // Children are checked with childBudget each; a child left unresolved is simply
    // not listed, so `winning` is a subset of the true winning set (never wrong).
    struct Label {
        int value;
        std::vector<Move> winning;
    };
    Label label(const Pos& p, uint64_t rootBudget, uint64_t childBudget) {
        Label L{0, {}};
        bool v0 = verbose;
        verbose = false;
        Gen g;
        genSafe(p, g);
        if (g.win) {
            L.value = 1;
            BB occ = p.me | p.op, L3 = p.lvl(3);
            for (BB x = p.me & p.lvl(2); x; x &= x - 1)
                for (BB t = T.adj[lsb(x)] & L3 & ~occ; t; t &= t - 1)
                    L.winning.push_back(Move{(uint8_t)lsb(x), (uint8_t)lsb(t), (uint8_t)lsb(x)});
            verbose = v0;
            return L;
        }
        budget = rootBudget;
        L.value = solve(p, 1e9);
        if (L.value == 1) {
            budget = childBudget;
            for (int i = 0; i < g.n; i++) {
                Pos c = makeChild(p, g.m[i]);
                uint32_t pn, dn;
                bool known = tt.lookup(canonicalKey(makeRoot(c)), pn, dn);
                if ((known && dn == 0) || ((!known || pn != 0) && solve(c, 1e9) == -1))
                    L.winning.push_back(g.m[i]);
            }
        }
        budget = 0;
        verbose = v0;
        return L;
    }

    // Value of a position from the TT, solving it (re-proving) if it fell out.
    int value(const Pos& p, double seconds = 1e9) {
        Node n = makeRoot(p);
        uint32_t pn, dn;
        if (tt.lookup(canonicalKey(n), pn, dn) && (pn == 0 || dn == 0)) return pn == 0 ? 1 : -1;
        bool v = verbose;
        verbose = false;
        int r = solve(p, seconds);
        verbose = v;
        return r;
    }
};

Overwriting solver.h


In [15]:
%%writefile az.h
// AlphaZero layer on top of engine.h / solver.h:
//   * compact state <-> Pos, action encoding, D4 action permutations
//   * network input planes (NHWC, 5x5x15)
//   * threaded solver labelling and Phase-0 position generation
//   * SelfPlay: many games in lock-step; each prepare() call returns one leaf per
//     game for the network, apply() expands/backs up. Uses forced-move pruning,
//     MCTS-Solver proof propagation and an exact df-pn root solve late in the game.
#pragma once
#include "solver.h"
#include <atomic>
#include <memory>
#include <mutex>
#include <thread>
#include <vector>

namespace az {

constexpr int NPLANES = 15;
constexpr int NMOVE = 128;    // move+build actions: 64*k + 8*dirMove + dirBuild
constexpr int NACT = 153;     // + 25 placement actions (reserved, always masked for now)
constexpr int STATE_LEN = 29; // 25 heights, me0 < me1, op0 < op1 (cell indices)

const int DR[8] = {-1, -1, -1, 0, 0, 1, 1, 1};
const int DC[8] = {-1, 0, 1, -1, 1, -1, 0, 1};

struct Tabs {
    int dirOf[25][25];  // direction index from a to b, -1 if not adjacent
    int dirMap[8][8];   // dirMap[s][d]: direction d after symmetry s
};
inline Tabs A;

inline void init() {
    initTables();
    for (int a = 0; a < 25; a++)
        for (int b = 0; b < 25; b++) {
            A.dirOf[a][b] = -1;
            for (int d = 0; d < 8; d++)
                if (a / 5 + DR[d] == b / 5 && a % 5 + DC[d] == b % 5) A.dirOf[a][b] = d;
        }
    for (int s = 0; s < 8; s++)
        for (int d = 0; d < 8; d++) {
            int c = 12, n = (2 + DR[d]) * 5 + (2 + DC[d]);
            A.dirMap[s][d] = A.dirOf[T.sym[s][c]][T.sym[s][n]];
        }
}

// ---- state conversion ----------------------------------------------------------
inline Pos fromState(const int8_t* s) {
    Pos p;
    for (int c = 0; c < 25; c++)
        for (int i = 0; i < s[c]; i++) p.raise(c);
    p.me = bit(s[25]) | bit(s[26]);
    p.op = bit(s[27]) | bit(s[28]);
    return p;
}
inline void toState(const Pos& p, int8_t* s) {
    for (int c = 0; c < 25; c++) s[c] = (int8_t)p.height(c);
    s[25] = (int8_t)lsb(p.me);
    s[26] = (int8_t)lsb(p.me & (p.me - 1));
    s[27] = (int8_t)lsb(p.op);
    s[28] = (int8_t)lsb(p.op & (p.op - 1));
}
inline Pos transformPos(const Pos& p, int s) {
    Pos q;
    for (int c = 0; c < 25; c++)
        for (int i = 0; i < p.height(c); i++) q.raise(T.sym[s][c]);
    for (BB x = p.me; x; x &= x - 1) q.me |= bit(T.sym[s][lsb(x)]);
    for (BB x = p.op; x; x &= x - 1) q.op |= bit(T.sym[s][lsb(x)]);
    return q;
}

// ---- actions ---------------------------------------------------------------------
// Worker k = 0 is the mover's worker on the lower cell index. An immediate step-up
// win has no build; it is encoded with build = the vacated cell (b = f).
inline int encodeMove(const Pos& p, Move m) {
    int k = (m.f == lsb(p.me)) ? 0 : 1;
    return 64 * k + 8 * A.dirOf[m.f][m.t] + A.dirOf[m.t][m.b];
}
// perm[a] = index of action a after applying symmetry s to position p.
inline void actionPerm(const Pos& p, int s, int32_t* perm) {
    int w0 = lsb(p.me), w1 = lsb(p.me & (p.me - 1));
    int nk0 = T.sym[s][w0] < T.sym[s][w1] ? 0 : 1;
    int nk[2] = {nk0, 1 - nk0};
    for (int k = 0; k < 2; k++)
        for (int dm = 0; dm < 8; dm++)
            for (int db = 0; db < 8; db++)
                perm[64 * k + 8 * dm + db] = 64 * nk[k] + 8 * A.dirMap[s][dm] + A.dirMap[s][db];
}
// Legal (safe) actions. All zero means the mover is lost.
inline int legalMask(const Pos& p, uint8_t* mask) {
    memset(mask, 0, NMOVE);
    Gen g;
    genSafe(p, g);
    int n = 0;
    if (g.win) {
        BB occ = p.me | p.op, L3 = p.lvl(3);
        for (BB x = p.me & p.lvl(2); x; x &= x - 1)
            for (BB t = T.adj[lsb(x)] & L3 & ~occ; t; t &= t - 1) {
                mask[encodeMove(p, Move{(uint8_t)lsb(x), (uint8_t)lsb(t), (uint8_t)lsb(x)})] = 1;
                n++;
            }
        return n;
    }
    for (int i = 0; i < g.n; i++) mask[encodeMove(p, g.m[i])] = 1;
    return g.n;
}
// Decode an action index into a Move (no legality check).
inline Move decode(const Pos& p, int a) {
    int k = a / 64, dm = (a / 8) % 8, db = a % 8;
    int f = k == 0 ? lsb(p.me) : lsb(p.me & (p.me - 1));
    int t = f + DR[dm] * 5 + DC[dm];
    int b = t + DR[db] * 5 + DC[db];
    return Move{(uint8_t)f, (uint8_t)t, (uint8_t)b};
}

// ---- input planes (NHWC: out[cell * NPLANES + plane]) ------------------------------
inline BB threatCells(const Pos& p, BB workers) {
    BB occ = p.me | p.op, adj = 0;
    for (BB x = workers & p.lvl(2); x; x &= x - 1) adj |= T.adj[lsb(x)];
    return adj & p.lvl(3) & ~occ;
}
inline void encodePlanes(const Pos& p, float* out) {
    memset(out, 0, sizeof(float) * 25 * NPLANES);
    BB myT = threatCells(p, p.me), opT = threatCells(p, p.op);
    BB build = p.buildable() & ~(p.me | p.op);
    float sc[5] = {p.blocks() / 68.0f, (LIMIT1 - pc(p.h1)) / 22.0f, (LIMIT2 - pc(p.h2)) / 18.0f,
                   (LIMIT3 - pc(p.h3)) / 14.0f, 1.0f /* move phase */};
    for (int c = 0; c < 25; c++) {
        float* o = out + c * NPLANES;
        o[p.height(c)] = 1;
        o[5] = (p.me >> c) & 1;
        o[6] = (p.op >> c) & 1;
        o[7] = (myT >> c) & 1;
        o[8] = (opT >> c) & 1;
        o[9] = (build >> c) & 1;
        for (int i = 0; i < 5; i++) o[10 + i] = sc[i];
    }
}

// ---- random positions --------------------------------------------------------------
inline Pos randomPlacement(std::mt19937_64& rng) {
    int cells[25];
    for (int i = 0; i < 25; i++) cells[i] = i;
    std::shuffle(cells, cells + 25, rng);
    Pos p;
    p.me = bit(cells[0]) | bit(cells[1]);
    p.op = bit(cells[2]) | bit(cells[3]);
    return p;
}
// Random safe play to exactly `blocks` blocks; false if the game ended first.
inline bool randomPosition(std::mt19937_64& rng, int blocks, Pos& out) {
    Pos p = randomPlacement(rng);
    for (;;) {
        Gen g;
        genSafe(p, g);
        if (g.win || g.n == 0) return false;
        if (p.blocks() >= blocks) { out = p; return true; }
        p = makeChild(p, g.m[rng() % g.n]);
    }
}

// ---- threaded labelling ---------------------------------------------------------------
struct LabelOut {
    int8_t value;       // +1 / -1 / 0 (unresolved)
    uint8_t win[NMOVE]; // proven winning actions (subset of the true set)
};
inline void labelOne(Solver& S, const Pos& p, uint64_t rootBudget, uint64_t childBudget,
                     LabelOut& out) {
    Solver::Label L = S.label(p, rootBudget, childBudget);
    out.value = (int8_t)L.value;
    memset(out.win, 0, NMOVE);
    for (Move m : L.winning) out.win[encodeMove(p, m)] = 1;
}
inline std::unique_ptr<Solver> makeSolver(double ttGb) {
    auto S = std::make_unique<Solver>();
    S->verbose = false;
    S->tt.alloc(ttGb);
    return S;
}
inline void labelBatch(const std::vector<Pos>& ps, uint64_t rootBudget, uint64_t childBudget,
                       int threads, double ttGb, std::vector<LabelOut>& out) {
    out.resize(ps.size());
    std::atomic<size_t> next{0};
    std::vector<std::thread> th;
    for (int t = 0; t < threads; t++)
        th.emplace_back([&]() {
            auto S = makeSolver(ttGb);
            for (size_t i; (i = next++) < ps.size();) labelOne(*S, ps[i], rootBudget, childBudget, out[i]);
        });
    for (auto& t : th) t.join();
}
// Phase 0: n resolved positions with blocks uniform in [minB, maxB].
inline void generateLabeled(int n, int minB, int maxB, uint64_t rootBudget, uint64_t childBudget,
                            int threads, double ttGb, uint64_t seed, std::vector<Pos>& ps,
                            std::vector<LabelOut>& out) {
    ps.resize(n);
    out.resize(n);
    std::atomic<int> next{0};
    std::vector<std::thread> th;
    for (int t = 0; t < threads; t++)
        th.emplace_back([&, t]() {
            auto S = makeSolver(ttGb);
            std::mt19937_64 rng(seed * 1000003 + t);
            for (;;) {
                if (next.load() >= n) return;
                Pos p;
                int target = minB + (int)(rng() % (maxB - minB + 1));
                if (!randomPosition(rng, target, p)) continue;
                LabelOut lo;
                labelOne(*S, p, rootBudget, childBudget, lo);
                if (lo.value == 0) continue;
                int i = next++;
                if (i >= n) return;
                ps[i] = p;
                out[i] = lo;
            }
        });
    for (auto& t : th) t.join();
}

// ---- self-play MCTS ---------------------------------------------------------------------
struct Config {
    int nGames = 256;           // games played in lock-step (= inference batch)
    int sims = 200;             // simulations per move
    float cpuct = 1.5f;
    float dirAlpha = 0.3f, dirFrac = 0.25f;
    bool noise = true;          // Dirichlet noise at the root (off for arena games)
    int tempPlies = 8;          // sample moves proportional to visits for this many plies
    int labelMinBlocks = 22;    // exact root solve (and label) from this many blocks
    uint64_t rootBudget = 20000, childBudget = 2000;
    int threads = 4;            // solver threads
    double ttGb = 0.125;        // TT per solver thread
    int maxGames = 1 << 30;     // stop starting new games after this many
    bool arena = false;         // two networks: net id per side, colours alternate
    uint64_t seed = 1;
};

struct Sample {
    int8_t state[STATE_LEN];
    float pi[NMOVE];
    float q;          // root value estimate (mover's view)
    int8_t exact;     // 1 if v is a proven value
    int8_t v;         // proven value (+1/-1) when exact
    int8_t player;    // 0 = first player, 1 = second player
    float z;          // final outcome for `player`, filled at game end
};

struct Edge {
    Move m;
    int16_t a;        // action index
    int child = -1;
    float P = 0, W = 0;  // W summed from the parent's mover perspective
    int N = 0;
};
struct MNode {
    Pos p;
    int first = 0, n = 0;  // edges [first, first+n)
    int parent = -1, pedge = -1;
    int N = 0;
    int8_t status = 0;     // +1 mover wins, -1 mover loses (proven)
    bool expanded = false;
};

class SelfPlay {
   public:
    Config cfg;
    explicit SelfPlay(const Config& c) : cfg(c), games(c.nGames) {
        rng.seed(c.seed);
        for (int t = 0; t < cfg.threads; t++) solvers.push_back(makeSolver(cfg.ttGb));
        for (int i = 0; i < cfg.nGames; i++) startGame(i);
    }

    // Advance all games until each needs a network evaluation (or is finished).
    // Fills planes[count][25][NPLANES], gameIdx[count], netId[count]; returns count.
    int prepare(float* planes, int32_t* gameIdx, int32_t* netId) {
        for (;;) {
            std::vector<int> needLabel;
            for (int i = 0; i < (int)games.size(); i++) {
                int st = advance(i);
                if (st == NEED_LABEL) needLabel.push_back(i);
            }
            if (needLabel.empty()) break;
            runLabels(needLabel);
            rootProven = provenCount.load();
        }
        int cnt = 0;
        for (int i = 0; i < (int)games.size(); i++) {
            Game& g = games[i];
            if (!g.active || g.leaf < 0) continue;
            encodePlanes(g.nodes[g.leaf].p, planes + (size_t)cnt * 25 * NPLANES);
            gameIdx[cnt] = i;
            netId[cnt] = g.netOf[g.ply & 1];
            cnt++;
        }
        return cnt;
    }

    // logits[count][NACT], values[count] for the leaves returned by prepare().
    void apply(int count, const int32_t* gameIdx, const float* logits, const float* values) {
        leafEvals += count;
        for (int j = 0; j < count; j++) {
            Game& g = games[gameIdx[j]];
            int leaf = g.leaf;
            expand(g, leaf, logits + (size_t)j * NACT);
            g.leaf = -1;
            backup(g, leaf, values[j]);
        }
    }

    // Finished-game samples since the last call.
    std::vector<Sample> harvest() {
        std::vector<Sample> out;
        out.swap(done);
        return out;
    }
    int gamesStarted() const { return started; }
    size_t pendingSamples() const { return done.size(); }
    // counters for logging
    long movesPlayed = 0, rootSolves = 0, rootProven = 0, solverMoves = 0, leafEvals = 0;
    int gamesFinished() const { return finished; }
    int activeGames() const {
        int a = 0;
        for (auto& g : games) a += g.active;
        return a;
    }
    // Arena results: wins by net 0 / net 1, and first-player wins.
    int netWins[2] = {0, 0};
    int firstPlayerWins = 0;
    std::vector<int> gameLengths;

   private:
    enum { NEED_EVAL, NEED_LABEL, IDLE };
    struct Game {
        bool active = false;
        Pos pos;
        int ply = 0;
        int netOf[2] = {0, 0};  // network id playing first / second player
        std::vector<MNode> nodes;
        std::vector<Edge> edges;
        int leaf = -1;
        bool labelled = false;
        LabelOut label{};
        std::vector<Sample> samples;
    };
    std::vector<Game> games;
    std::vector<std::unique_ptr<Solver>> solvers;
    std::vector<Sample> done;
    std::mt19937_64 rng;
    int started = 0, finished = 0;
    std::atomic<long> provenCount{0};

    void startGame(int i) {
        Game& g = games[i];
        g = Game();
        if (started >= cfg.maxGames) return;
        g.active = true;
        g.pos = randomPlacement(rng);
        if (cfg.arena) {  // alternate colours between the two networks
            g.netOf[0] = started & 1;
            g.netOf[1] = 1 - g.netOf[0];
        }
        started++;
    }

    void finishGame(int i, int winnerPlayer) {
        Game& g = games[i];
        for (Sample& s : g.samples) {
            s.z = s.player == winnerPlayer ? 1.0f : -1.0f;
            done.push_back(s);
        }
        netWins[g.netOf[winnerPlayer]]++;
        firstPlayerWins += winnerPlayer == 0;
        gameLengths.push_back(g.ply);
        finished++;
        startGame(i);
    }

    void record(Game& g, const float* pi, float q, int exact, int v) {
        if (!cfg.arena) {
            Sample s;
            toState(g.pos, s.state);
            memcpy(s.pi, pi, sizeof(s.pi));
            s.q = q;
            s.exact = (int8_t)exact;
            s.v = (int8_t)v;
            s.player = (int8_t)(g.ply & 1);
            s.z = 0;
            g.samples.push_back(s);
        }
    }

    // Play move m in game i; returns true if the game ended.
    bool play(int i, Move m) {
        movesPlayed++;
        Game& g = games[i];
        int mover = g.ply & 1;
        if (g.pos.height(m.f) == 2 && g.pos.height(m.t) == 3) {  // step up: win
            g.ply++;
            finishGame(i, mover);
            return true;
        }
        g.pos = makeChild(g.pos, m);
        g.ply++;
        g.nodes.clear();
        g.edges.clear();
        g.labelled = false;
        return false;
    }

    void runLabels(const std::vector<int>& idx) {
        rootSolves += (long)idx.size();
        std::atomic<size_t> next{0};
        std::vector<std::thread> th;
        int nt = std::min<int>(cfg.threads, (int)idx.size());
        for (int t = 0; t < nt; t++)
            th.emplace_back([&, t]() {
                for (size_t k; (k = next++) < idx.size();) {
                    Game& g = games[idx[k]];
                    labelOne(*solvers[t], g.pos, cfg.rootBudget, cfg.childBudget, g.label);
                    g.labelled = true;
                    if (g.label.value != 0) provenCount++;
                }
            });
        for (auto& t : th) t.join();
    }

    int newNode(Game& g, const Pos& p, int parent, int pedge) {
        MNode n;
        n.p = p;
        n.parent = parent;
        n.pedge = pedge;
        Gen gen;
        genSafe(p, gen);
        if (gen.win) n.status = 1;
        else if (gen.n == 0) n.status = -1;
        else {
            n.first = (int)g.edges.size();
            n.n = gen.n;
            for (int k = 0; k < gen.n; k++) {
                Edge e;
                e.m = gen.m[k];
                e.a = (int16_t)encodeMove(p, gen.m[k]);
                g.edges.push_back(e);
            }
        }
        g.nodes.push_back(n);
        return (int)g.nodes.size() - 1;
    }

    // Returns NEED_EVAL (g.leaf set), NEED_LABEL or IDLE.
    int advance(int i) {
        for (;;) {
            Game& g = games[i];
            if (!g.active) return IDLE;
            if (g.leaf >= 0) return NEED_EVAL;
            if (g.nodes.empty()) {
                // New move. Exact root solve late in the game.
                if (g.pos.blocks() >= cfg.labelMinBlocks && !g.labelled) return NEED_LABEL;
                int root = newNode(g, g.pos, -1, -1);
                MNode& r = g.nodes[root];
                if (r.status == -1) { finishGame(i, 1 - (g.ply & 1)); continue; }  // no safe move
                bool labelWins = false;  // root proven won AND at least one winning move known
                if (g.labelled && g.label.value == 1)
                    for (int a = 0; a < NMOVE && !labelWins; a++) labelWins = g.label.win[a];
                if (r.status == 1 || labelWins) {
                    playProvenWin(i);
                    continue;
                }
                g.leaf = root;  // root itself needs a network evaluation first
                return NEED_EVAL;
            }
            MNode& root = g.nodes[0];
            if (root.status != 0 || root.N >= cfg.sims) {
                if (root.status == 1) { playProvenWin(i); continue; }
                chooseAndPlay(i);
                continue;
            }
            // One simulation: select down to an unexpanded node.
            int x = 0;
            for (;;) {
                MNode& n = g.nodes[x];
                if (n.status != 0) {  // proven / terminal: exact backup
                    backup(g, x, (float)n.status);
                    break;
                }
                if (!n.expanded) {  // freshly created child: needs the network
                    g.leaf = x;
                    return NEED_EVAL;
                }
                int e = select(g, x);
                if (e < 0) {  // every move loses: node is lost
                    g.nodes[x].status = -1;
                    propagate(g, g.nodes[x].parent);
                    continue;
                }
                Edge& ed = g.edges[e];
                if (ed.child < 0) {
                    Pos cp = makeChild(g.nodes[x].p, ed.m);
                    int c = newNode(g, cp, x, e);
                    g.edges[e].child = c;
                    if (g.nodes[c].status != 0) propagate(g, x);
                }
                x = g.edges[e].child;
            }
        }
    }

    // PUCT over edges, skipping moves into positions proven won for the opponent;
    // a move into a proven-lost position is taken immediately.
    int select(Game& g, int x) {
        MNode& n = g.nodes[x];
        float sq = std::sqrt((float)std::max(1, n.N));
        int best = -1;
        float bestV = -1e30f;
        for (int k = n.first; k < n.first + n.n; k++) {
            Edge& e = g.edges[k];
            if (e.child >= 0) {
                int st = g.nodes[e.child].status;
                if (st == -1) return k;
                if (st == 1) continue;
            }
            float q = e.N ? e.W / e.N : 0.0f;
            float u = cfg.cpuct * e.P * sq / (1 + e.N);
            if (q + u > bestV) { bestV = q + u; best = k; }
        }
        return best;
    }

    // MCTS-Solver: a node is won if some child is lost for its mover, and lost if
    // every child is won for its mover.
    void propagate(Game& g, int x) {
        while (x >= 0) {
            MNode& n = g.nodes[x];
            if (n.status != 0) return;
            bool anyLoss = false, allWin = true;
            for (int k = n.first; k < n.first + n.n; k++) {
                int c = g.edges[k].child;
                int st = c >= 0 ? g.nodes[c].status : 0;
                if (st == -1) anyLoss = true;
                if (st != 1) allWin = false;
            }
            if (anyLoss) n.status = 1;
            else if (allWin) n.status = -1;
            else return;
            x = n.parent;
        }
    }

    void expand(Game& g, int x, const float* logits) {
        MNode& n = g.nodes[x];
        n.expanded = true;
        float mx = -1e30f;
        for (int k = n.first; k < n.first + n.n; k++) mx = std::max(mx, logits[g.edges[k].a]);
        float sum = 0;
        for (int k = n.first; k < n.first + n.n; k++) {
            float p = std::exp(logits[g.edges[k].a] - mx);
            g.edges[k].P = p;
            sum += p;
        }
        for (int k = n.first; k < n.first + n.n; k++) g.edges[k].P /= sum;
        if (x == 0 && cfg.noise && n.n > 0) {
            std::gamma_distribution<float> gam(cfg.dirAlpha, 1.0f);
            std::vector<float> d(n.n);
            float ds = 0;
            for (float& v : d) { v = gam(rng); ds += v; }
            for (int k = 0; k < n.n; k++) {
                Edge& e = g.edges[n.first + k];
                e.P = (1 - cfg.dirFrac) * e.P + cfg.dirFrac * d[k] / std::max(ds, 1e-12f);
            }
        }
    }

    // v is from the mover's perspective at node x.
    void backup(Game& g, int x, float v) {
        for (;;) {
            MNode& n = g.nodes[x];
            n.N++;
            if (n.parent < 0) break;
            Edge& e = g.edges[n.pedge];
            e.N++;
            e.W += -v;
            v = -v;
            x = n.parent;
        }
    }

    void playProvenWin(int i) {
        Game& g = games[i];
        float pi[NMOVE] = {0};
        std::vector<int> wins;
        uint8_t mask[NMOVE];
        if (g.labelled && g.label.value == 1)
            for (int a = 0; a < NMOVE; a++)
                if (g.label.win[a]) wins.push_back(a);
        if (!wins.empty()) {
        } else if (legalMask(g.pos, mask) && canWinNow(g.pos, g.pos.me)) {
            for (int a = 0; a < NMOVE; a++)
                if (mask[a]) wins.push_back(a);
        } else if (!g.nodes.empty()) {  // proven by the tree (MCTS-Solver)
            MNode& r = g.nodes[0];
            for (int k = r.first; k < r.first + r.n; k++) {
                int c = g.edges[k].child;
                if (c >= 0 && g.nodes[c].status == -1) wins.push_back(g.edges[k].a);
            }
        }
        if (wins.empty()) {  // should not happen; fall back to the search
            chooseAndPlay(i);
            return;
        }
        solverMoves++;
        for (int a : wins) pi[a] = 1.0f / wins.size();
        record(g, pi, 1.0f, 1, 1);
        int a = wins[rng() % wins.size()];
        play(i, decode(g.pos, a));
    }

    void chooseAndPlay(int i) {
        Game& g = games[i];
        MNode& r = g.nodes[0];
        float pi[NMOVE] = {0};
        int total = 0;
        float wsum = 0;
        for (int k = r.first; k < r.first + r.n; k++) {
            total += g.edges[k].N;
            wsum += g.edges[k].W;
        }
        // Candidate moves: exclude moves proven to lose unless all of them do.
        std::vector<int> cand;
        for (int k = r.first; k < r.first + r.n; k++) {
            int c = g.edges[k].child;
            if (!(c >= 0 && g.nodes[c].status == 1)) cand.push_back(k);
        }
        if (cand.empty())
            for (int k = r.first; k < r.first + r.n; k++) cand.push_back(k);
        for (int k : cand) pi[g.edges[k].a] = (float)g.edges[k].N;
        float s = 0;
        for (float v : pi) s += v;
        if (s <= 0) {
            for (int k : cand) pi[g.edges[k].a] = 1;
            s = (float)cand.size();
        }
        for (float& v : pi) v /= s;
        float q = total ? wsum / total : 0.0f;
        bool exact = (g.labelled && g.label.value != 0) || r.status != 0;
        int v = r.status != 0 ? r.status : g.label.value;
        record(g, pi, q, exact, exact ? v : 0);

        int pick;
        if (g.ply < cfg.tempPlies) {
            std::uniform_real_distribution<float> U(0, 1);
            float x = U(rng), acc = 0;
            pick = cand.back();
            for (int k : cand) {
                acc += pi[g.edges[k].a];
                if (x <= acc) { pick = k; break; }
            }
        } else {
            pick = cand[0];
            for (int k : cand)
                if (g.edges[k].N > g.edges[pick].N) pick = k;
        }
        play(i, g.edges[pick].m);
    }
};

}  // namespace az

Overwriting az.h


In [16]:
%%writefile agent.h
// Single-position search for playing and analysis.
//
// Same MCTS as self-play (PUCT, forced-move pruning, MCTS-Solver proofs, network
// priors/values) but for one position at a time, with:
//   * batched leaf collection using virtual loss, so one network call evaluates
//     many leaves;
//   * an optional exact df-pn solve of the root (value + every proven winning move);
//   * full introspection: per-move visits, value, prior, proof status and the
//     principal variation, for visualising what the agent is "thinking".
#pragma once
#include "az.h"
#include <climits>
#include <cmath>
#include <set>

namespace az {

struct SearchParams {
    float cpuct = 1.5f;
    uint64_t rootBudget = 200000;   // df-pn expansions for the root solve (0 = off)
    uint64_t childBudget = 2000;    // per move, to list the winning moves of a won root
    double ttGb = 0.25;
};

struct MoveStat {
    Move m;
    int a;              // action index (network policy index)
    bool stepUp;        // immediate win by stepping up to level 3
    int N;
    float Q;            // mean value for the root mover (-1..1), 0 if unvisited
    float P;            // network prior
    int proof;          // +1 proven win for the root mover, -1 proven loss, 0 unknown
    std::vector<Move> pv;  // continuation after this move (alternating sides)
    bool pvEndsInWin;   // last pv entry is a step-up win
};

class Search {
   public:
    Search(const Pos& root, const SearchParams& sp) : prm(sp) {
        rootPos = root;
        // Immediate step-up wins and positions with no safe move are decided at once.
        Gen g;
        genSafe(root, g);
        if (g.win) {
            rootStatus = 1;
            BB occ = root.me | root.op, L3 = root.lvl(3);
            for (BB x = root.me & root.lvl(2); x; x &= x - 1)
                for (BB t = T.adj[lsb(x)] & L3 & ~occ; t; t &= t - 1)
                    winNow.push_back(Move{(uint8_t)lsb(x), (uint8_t)lsb(t), (uint8_t)lsb(x)});
            return;
        }
        newNode(root, -1, -1);
        if (g.n == 0) { rootStatus = -1; nodes[0].status = -1; return; }
        if (prm.rootBudget) {
            solver = makeSolver(prm.ttGb);
            LabelOut lo;
            labelOne(*solver, root, prm.rootBudget, prm.childBudget, lo);
            solverValue = lo.value;
            if (lo.value == 1)
                for (int a = 0; a < NMOVE; a++)
                    if (lo.win[a]) solverWins.insert(a);
            if (lo.value != 0) rootStatus = lo.value;
            solver.reset();  // the transposition table is only needed for the root solve
        }
    }

    // Collect up to maxLeaves leaves that need the network; writes their planes.
    // Simulations ending in proven/terminal nodes are backed up immediately.
    int prepare(float* planes, int maxLeaves) {
        pending.clear();
        if (done()) return 0;
        int guard = 0;
        while ((int)pending.size() < maxLeaves && guard++ < maxLeaves * 4) {
            if (done()) break;  // proven by the tree
            std::vector<int> path;  // edge indices from the root
            int x = 0;
            bool collided = false;
            for (;;) {
                MNode& n = nodes[x];
                if (n.status != 0) {  // terminal / proven: exact value
                    addVirtual(path);
                    backup(path, (float)n.status);
                    sims++;
                    break;
                }
                if (!n.expanded) {
                    if (pendingNode.count(x)) { collided = true; break; }
                    addVirtual(path);
                    pendingNode.insert(x);
                    pending.push_back({x, path});
                    encodePlanes(n.p, planes + (size_t)(pending.size() - 1) * 25 * NPLANES);
                    break;
                }
                int e = select(x);
                if (e < 0) {  // every move loses: node is lost
                    nodes[x].status = -1;
                    propagate(nodes[x].parent);
                    continue;  // re-handle as a proven node
                }
                path.push_back(e);
                if (edges[e].child < 0) {
                    int c = newNode(makeChild(nodes[x].p, edges[e].m), x, e);
                    edges[e].child = c;
                    if (nodes[c].status != 0) propagate(x);
                }
                x = edges[e].child;
            }
            if (collided) break;
        }
        return (int)pending.size();
    }

    // logits[count][NACT], values[count] for the leaves of the last prepare().
    void apply(const float* logits, const float* values) {
        for (size_t j = 0; j < pending.size(); j++) {
            int x = pending[j].node;
            expand(x, logits + j * NACT);
            netV[x] = values[j];
            if (x == 0) rootNetV = values[j];
            backup(pending[j].path, values[j]);
            pendingNode.erase(x);
            sims++;
        }
        pending.clear();
    }

    // Nothing left to search: immediate win, no safe move, or the tree proved the root.
    bool done() const {
        if (winNow.size() || nodes.empty()) return true;
        const MNode& r = nodes[0];
        return r.status != 0 && (r.expanded || r.n == 0);
    }
    int simulations() const { return sims; }
    int rootN() const { return nodes.empty() ? 0 : nodes[0].N; }
    int status() const {  // +1 / -1 proven for the root mover (solver or tree), 0 unknown
        if (rootStatus) return rootStatus;
        return nodes.empty() ? 0 : nodes[0].status;
    }
    int solverResult() const { return solverValue; }
    float netValue() const { return rootNetV; }
    float searchValue() const {
        if (nodes.empty()) return winNow.size() ? 1.0f : 0.0f;
        const MNode& r = nodes[0];
        int n = 0;
        float w = 0;
        for (int k = r.first; k < r.first + r.n; k++) { n += edges[k].N; w += edges[k].W; }
        return n ? w / n : rootNetV;
    }
    size_t treeSize() const { return nodes.size(); }

    std::vector<MoveStat> stats(int pvDepth) const {
        std::vector<MoveStat> out;
        for (Move m : winNow) out.push_back(MoveStat{m, encodeMove(rootPos, m), true, 0, 1.0f, 1.0f, 1, {}, false});
        if (nodes.empty()) return out;
        const MNode& r = nodes[0];
        for (int k = r.first; k < r.first + r.n; k++) {
            const Edge& e = edges[k];
            MoveStat s{e.m, e.a, false, e.N, e.N ? e.W / e.N : 0.0f, e.P, 0, {}, false};
            if (e.child >= 0) s.proof = -nodes[e.child].status;
            if (solverValue == 1 && solverWins.count(e.a)) s.proof = 1;
            if (solverValue == -1) s.proof = -1;
            if (e.child >= 0) pv(e.child, pvDepth, s.pv, s.pvEndsInWin);
            out.push_back(std::move(s));
        }
        std::sort(out.begin(), out.end(), [](const MoveStat& a, const MoveStat& b) {
            if (a.stepUp != b.stepUp) return a.stepUp;
            if (a.N != b.N) return a.N > b.N;
            return a.P > b.P;
        });
        return out;
    }

    // Move to play: an immediate win; else a proven win (most visited); else the most
    // visited move that is not proven lost; ties broken by the prior.
    MoveStat best() const {
        auto st = stats(0);
        if (st.empty()) return MoveStat{Move{0, 0, 0}, -1, false, 0, 0, 0, 0, {}, false};
        for (auto& s : st) if (s.stepUp) return s;
        for (auto& s : st) if (s.proof == 1) return s;
        for (auto& s : st) if (s.proof != -1) return s;
        return st[0];
    }


    // ---- Step-by-step search (for teaching; no virtual loss, one simulation) -------
    // stepSelect: walk down by PUCT to a leaf. Returns 1 if the leaf needs the network
    // (its planes are written to `planes`), 2 if its value is already known (terminal or
    // proven), 0 if the search is finished. stepExpand / stepExpandKnown evaluate it,
    // stepBackup adds the value along the path.
    int stepSelect(float* planes) {
        stepPath.clear();
        stepLeaf = -1;
        stepState = 0;
        if (done() || !pending.empty()) return 0;
        int x = 0;
        for (;;) {
            MNode& n = nodes[x];
            if (n.status != 0) { stepLeaf = x; stepState = 2; stepValue = (float)n.status; return 2; }
            if (!n.expanded) {
                stepLeaf = x;
                stepState = 1;
                encodePlanes(n.p, planes);
                return 1;
            }
            int e = select(x);
            if (e < 0) { nodes[x].status = -1; propagate(nodes[x].parent); continue; }
            stepPath.push_back(e);
            if (edges[e].child < 0) {
                int c = newNode(makeChild(nodes[x].p, edges[e].m), x, e);
                edges[e].child = c;
                if (nodes[c].status != 0) propagate(x);
            }
            x = edges[e].child;
        }
    }
    void stepExpand(const float* logits, float value) {
        if (stepState != 1) return;
        expand(stepLeaf, logits);
        netV[stepLeaf] = value;
        if (stepLeaf == 0) rootNetV = value;
        stepValue = value;
        stepState = 3;
    }
    void stepExpandKnown() { if (stepState == 2) stepState = 3; }
    void stepBackup() {
        if (stepState != 3) return;
        nodes[0].N++;
        float v = stepValue;
        for (int i = (int)stepPath.size() - 1; i >= 0; i--) {
            Edge& e = edges[stepPath[i]];
            e.N++;
            e.W += -v;
            nodes[e.child].N++;
            v = -v;
        }
        sims++;
        stepState = 4;  // backed up; the path stays highlighted until the next select
    }
    int stepPhase() const { return stepState; }
    int stepLeafNode() const { return stepLeaf; }
    float stepLeafValue() const { return stepValue; }
    std::vector<int> stepNodes() const {  // node ids on the current step path
        std::vector<int> out;
        if (stepState == 0 || nodes.empty()) return out;
        out.push_back(0);
        for (int e : stepPath) out.push_back(edges[e].child);
        return out;
    }

    // ---- Tree export for the visualiser ----------------------------------------------
    struct TreeItem {
        int id;            // node index, or -(edge + 1) for a move that has no node yet
        int parent;        // id of the parent item, INT_MIN for the displayed root
        int depth;         // plies from the search root
        int N;             // visits (virtual loss removed)
        float Q;           // mean value from the parent mover's view (edge), or the root mover's view at the search root
        float P;           // network prior of the incoming move (1 for the root)
        float U;           // PUCT exploration bonus of the incoming move
        int status;        // +1 / -1 proven for the side to move at this node, 0 unknown
        bool expanded;     // evaluated by the network
        float netV;        // network value for the side to move at this node (NaN if not evaluated)
        bool hasMove;
        Move m;            // incoming move
        Pos p;             // position (side to move's perspective)
        bool onPV, onStep;
        int childCount;    // safe moves available at this node
    };

    std::vector<TreeItem> exportTree(int root, int maxNodes, float prunePct, int maxDepth, bool unvisited) const {
        std::vector<TreeItem> out;
        if (nodes.empty()) return out;
        if (root < 0 || root >= (int)nodes.size()) root = 0;
        std::set<int> pvSet, stepSet;
        for (int x = 0, guard = 0; x >= 0 && guard < 64; guard++) {
            pvSet.insert(x);
            const MNode& n = nodes[x];
            int best = -1;
            for (int k = n.first; k < n.first + n.n; k++)
                if (edges[k].child >= 0 && dispN(k) > 0 && (best < 0 || dispN(k) > dispN(best))) best = k;
            x = best < 0 ? -1 : edges[best].child;
        }
        for (int x : stepNodes()) stepSet.insert(x);
        struct Cand { int node, edge, parent, depth; float prio; };
        std::vector<Cand> cands;
        int rootDepth = depthOf[root];
        cands.push_back({root, nodes[root].pedge, INT_MIN, rootDepth, 1e30f});
        std::vector<int> stack{0};
        while (!stack.empty()) {
            Cand c = cands[stack.back()];
            stack.pop_back();
            if (c.node < 0 || c.depth - rootDepth >= maxDepth) continue;
            const MNode& n = nodes[c.node];
            if (!n.expanded) continue;
            std::vector<int> ks;
            int total = 0;
            for (int k = n.first; k < n.first + n.n; k++) { ks.push_back(k); total += dispN(k); }
            std::sort(ks.begin(), ks.end(), [&](int a, int b) {
                if (dispN(a) != dispN(b)) return dispN(a) > dispN(b);
                return edges[a].P > edges[b].P;
            });
            int acc = 0;
            for (int k : ks) {
                int ch = edges[k].child;
                bool forced = ch >= 0 && stepSet.count(ch);
                if (!forced && prunePct < 100 && total > 0 && acc >= prunePct / 100.0f * total) break;
                if (!forced && dispN(k) == 0 && !unvisited) continue;
                acc += dispN(k);
                float prio = forced ? 1e29f : dispN(k) > 0 ? (float)dispN(k) : edges[k].P * 0.999f;
                cands.push_back({ch, k, c.node, c.depth + 1, prio});
                if (ch >= 0) stack.push_back((int)cands.size() - 1);
            }
        }
        // global budget: most visited first, parents before children
        std::vector<int> order(cands.size());
        for (size_t i = 0; i < order.size(); i++) order[i] = (int)i;
        std::stable_sort(order.begin(), order.end(), [&](int a, int b) {
            if (cands[a].prio != cands[b].prio) return cands[a].prio > cands[b].prio;
            return cands[a].depth < cands[b].depth;
        });
        std::set<int> keptNodes;
        for (int i : order) {
            const Cand& c = cands[i];
            if ((int)out.size() >= std::max(1, maxNodes) && c.prio < 1e29f) break;
            if (c.parent != INT_MIN && !keptNodes.count(c.parent)) continue;
            TreeItem t{};
            t.depth = c.depth;
            t.parent = c.parent;
            t.hasMove = c.edge >= 0;
            if (c.edge >= 0) {
                const Edge& e = edges[c.edge];
                int pn = nodes[c.parent == INT_MIN ? nodes[c.node].parent : c.parent].N;
                t.m = e.m;
                t.N = dispN(c.edge);
                t.Q = t.N ? (e.W + vl[c.edge]) / t.N : 0.0f;
                t.P = e.P;
                t.U = prm.cpuct * e.P * std::sqrt((float)std::max(1, pn)) / (1 + t.N);
            } else {
                t.N = nodes[0].N - rootVl;
                t.Q = searchValue();
                t.P = 1;
                t.U = 0;
            }
            if (c.node >= 0) {
                const MNode& n = nodes[c.node];
                t.id = c.node;
                t.status = n.status;
                t.expanded = n.expanded;
                t.netV = netV[c.node];
                t.p = n.p;
                t.childCount = n.n;
                t.onPV = pvSet.count(c.node) > 0;
                t.onStep = stepSet.count(c.node) > 0;
                keptNodes.insert(c.node);
            } else {
                t.id = -(c.edge + 1);
                t.status = 0;
                t.expanded = false;
                t.netV = NAN;
                t.p = makeChild(nodes[c.parent].p, edges[c.edge].m);
                t.childCount = -1;
                t.onPV = t.onStep = false;
            }
            out.push_back(t);
        }
        return out;
    }
    size_t nodeCount() const { return nodes.size(); }

   private:
    struct Pending {
        int node;
        std::vector<int> path;
    };
    SearchParams prm;
    Pos rootPos;
    std::vector<MNode> nodes;
    std::vector<Edge> edges;
    std::vector<Pending> pending;
    std::set<int> pendingNode;
    std::vector<Move> winNow;
    std::unique_ptr<Solver> solver;
    std::set<int> solverWins;
    int solverValue = 0, rootStatus = 0, sims = 0;
    float rootNetV = 0;
    std::vector<uint16_t> vl;      // virtual losses currently on each edge
    int rootVl = 0;
    std::vector<float> netV;       // network value per node (side to move), NaN until evaluated
    std::vector<int> depthOf;      // plies from the root per node
    std::vector<int> stepPath;     // edges of the step-by-step simulation
    int stepLeaf = -1, stepState = 0;
    float stepValue = 0;
    int dispN(int e) const { return edges[e].N - vl[e]; }

    int newNode(const Pos& p, int parent, int pedge) {
        MNode n;
        n.p = p;
        n.parent = parent;
        n.pedge = pedge;
        Gen gen;
        genSafe(p, gen);
        if (gen.win) n.status = 1;
        else if (gen.n == 0) n.status = -1;
        else {
            n.first = (int)edges.size();
            n.n = gen.n;
            for (int k = 0; k < gen.n; k++) {
                Edge e;
                e.m = gen.m[k];
                e.a = (int16_t)encodeMove(p, gen.m[k]);
                edges.push_back(e);
            }
        }
        nodes.push_back(n);
        vl.resize(edges.size(), 0);
        netV.push_back(NAN);
        depthOf.push_back(parent < 0 ? 0 : depthOf[parent] + 1);
        return (int)nodes.size() - 1;
    }

    int select(int x) const {
        const MNode& n = nodes[x];
        float sq = std::sqrt((float)std::max(1, n.N));
        int best = -1;
        float bestV = -1e30f;
        for (int k = n.first; k < n.first + n.n; k++) {
            const Edge& e = edges[k];
            if (e.child >= 0) {
                int st = nodes[e.child].status;
                if (st == -1) return k;
                if (st == 1) continue;
            }
            float q = e.N ? e.W / e.N : 0.0f;
            float u = prm.cpuct * e.P * sq / (1 + e.N);
            if (q + u > bestV) { bestV = q + u; best = k; }
        }
        return best;
    }

    void propagate(int x) {
        while (x >= 0) {
            MNode& n = nodes[x];
            if (n.status != 0) return;
            bool anyLoss = false, allWin = true;
            for (int k = n.first; k < n.first + n.n; k++) {
                int c = edges[k].child;
                int st = c >= 0 ? nodes[c].status : 0;
                if (st == -1) anyLoss = true;
                if (st != 1) allWin = false;
            }
            if (anyLoss) n.status = 1;
            else if (allWin) n.status = -1;
            else return;
            x = n.parent;
        }
    }

    void expand(int x, const float* logits) {
        MNode& n = nodes[x];
        n.expanded = true;
        float mx = -1e30f;
        for (int k = n.first; k < n.first + n.n; k++) mx = std::max(mx, logits[edges[k].a]);
        float sum = 0;
        for (int k = n.first; k < n.first + n.n; k++) {
            float p = std::exp(logits[edges[k].a] - mx);
            edges[k].P = p;
            sum += p;
        }
        for (int k = n.first; k < n.first + n.n; k++) edges[k].P /= std::max(sum, 1e-20f);
    }

    // Virtual loss: count the visit now and score it as a loss for the edge's mover,
    // so the next selections in the same batch spread to other branches.
    void addVirtual(const std::vector<int>& path) {
        nodes[0].N++;
        rootVl++;
        for (int e : path) {
            edges[e].N++;
            edges[e].W -= 1.0f;
            vl[e]++;
            nodes[edges[e].child].N++;
        }
    }
    // Replace the virtual loss by the real value v (mover's perspective at the leaf).
    void backup(const std::vector<int>& path, float v) {
        for (int i = (int)path.size() - 1; i >= 0; i--) {
            edges[path[i]].W += 1.0f - v;  // edge value is from the parent's mover: -v
            vl[path[i]]--;
            v = -v;
        }
        rootVl--;
    }

    void pv(int x, int depth, std::vector<Move>& out, bool& endsInWin) const {
        endsInWin = false;
        for (int d = 0; d < depth && x >= 0; d++) {
            const MNode& n = nodes[x];
            if (n.status == 1 && n.n == 0) {  // mover steps up and wins
                BB occ = n.p.me | n.p.op, L3 = n.p.lvl(3);
                for (BB w = n.p.me & n.p.lvl(2); w; w &= w - 1) {
                    BB t = T.adj[lsb(w)] & L3 & ~occ;
                    if (t) {
                        out.push_back(Move{(uint8_t)lsb(w), (uint8_t)lsb(t), (uint8_t)lsb(w)});
                        endsInWin = true;
                        return;
                    }
                }
                return;
            }
            if (!n.expanded || n.n == 0) return;
            int best = -1;
            for (int k = n.first; k < n.first + n.n; k++) {
                if (edges[k].child >= 0 && nodes[edges[k].child].status == -1) { best = k; break; }
                if (edges[k].N > 0 && (best < 0 || edges[k].N > edges[best].N)) best = k;
            }
            if (best < 0) return;
            out.push_back(edges[best].m);
            x = edges[best].child;
        }
    }
};

// Every legal move+build for the side to move (no pruning). Step-up wins have b = 255.
inline std::vector<Move> allMoves(const Pos& p) {
    std::vector<Move> out;
    BB occ = p.me | p.op, bmAllowed = p.buildable();
    for (BB ws = p.me; ws; ws &= ws - 1) {
        int w = lsb(ws);
        for (BB d = T.adj[w] & ~occ & climbMask(p, p.height(w)); d; d &= d - 1) {
            int t = lsb(d);
            if (p.height(w) == 2 && p.height(t) == 3) {
                out.push_back(Move{(uint8_t)w, (uint8_t)t, 255});
                continue;
            }
            for (BB b = T.adj[t] & ~(occ ^ bit(w) ^ bit(t)) & bmAllowed; b; b &= b - 1)
                out.push_back(Move{(uint8_t)w, (uint8_t)t, (uint8_t)lsb(b)});
        }
    }
    return out;
}

}  // namespace az

Overwriting agent.h


In [17]:
%%writefile santorini_py.cpp
// Python bindings: python3 -m pybind11 --includes; built as module `santorini`.
#include <pybind11/numpy.h>
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>

#include "agent.h"

namespace py = pybind11;
using namespace az;
template <class T>
using arr = py::array_t<T, py::array::c_style | py::array::forcecast>;

static std::vector<Pos> toPositions(const arr<int8_t>& states) {
    if (states.ndim() != 2 || states.shape(1) != STATE_LEN)
        throw std::runtime_error("states must have shape [N, 29]");
    std::vector<Pos> ps(states.shape(0));
    const int8_t* s = states.data();
    for (size_t i = 0; i < ps.size(); i++) ps[i] = fromState(s + i * STATE_LEN);
    return ps;
}
static arr<int8_t> fromPositions(const std::vector<Pos>& ps) {
    arr<int8_t> out({(py::ssize_t)ps.size(), (py::ssize_t)STATE_LEN});
    for (size_t i = 0; i < ps.size(); i++) toState(ps[i], out.mutable_data() + i * STATE_LEN);
    return out;
}

PYBIND11_MODULE(santorini, m) {
    az::init();
    m.attr("NPLANES") = NPLANES;
    m.attr("NMOVE") = NMOVE;
    m.attr("NACT") = NACT;
    m.attr("STATE_LEN") = STATE_LEN;

    m.def("legal_mask", [](arr<int8_t> states) {
        auto ps = toPositions(states);
        arr<uint8_t> out({(py::ssize_t)ps.size(), (py::ssize_t)NMOVE});
        for (size_t i = 0; i < ps.size(); i++) legalMask(ps[i], out.mutable_data() + i * NMOVE);
        return out;
    }, "Safe (non-losing-in-one) actions per state, [N,128] uint8.");

    m.def("encode", [](arr<int8_t> states, arr<int32_t> syms) {
        auto ps = toPositions(states);
        py::ssize_t n = ps.size();
        if (syms.size() != n) throw std::runtime_error("syms must have length N");
        arr<float> planes({n, (py::ssize_t)5, (py::ssize_t)5, (py::ssize_t)NPLANES});
        arr<uint8_t> legal({n, (py::ssize_t)NMOVE});
        arr<int32_t> perm({n, (py::ssize_t)NMOVE});
        float* P = planes.mutable_data();
        uint8_t* L = legal.mutable_data();
        int32_t* Q = perm.mutable_data();
        const int32_t* S = syms.data();
        {
            py::gil_scoped_release nogil;
            for (py::ssize_t i = 0; i < n; i++) {
                Pos t = transformPos(ps[i], S[i]);
                encodePlanes(t, P + i * 25 * NPLANES);
                legalMask(t, L + i * NMOVE);
                actionPerm(ps[i], S[i], Q + i * NMOVE);
            }
        }
        return py::make_tuple(planes, legal, perm);
    }, "Planes [N,5,5,15], legal mask [N,128] and action permutation [N,128] after symmetry syms[i].\n"
       "A policy target pi over the original state maps to pi_t[perm[a]] = pi[a].");

    m.def("transform", [](arr<int8_t> states, arr<int32_t> syms) {
        auto ps = toPositions(states);
        for (size_t i = 0; i < ps.size(); i++) ps[i] = transformPos(ps[i], syms.data()[i]);
        return fromPositions(ps);
    }, "States after D4 symmetry syms[i] (0-3 rotations, 4-7 rotation+mirror).");

    m.def("apply_action", [](arr<int8_t> state, int a) {
        auto ps = toPositions(state.reshape({1, STATE_LEN}));
        Pos p = ps[0];
        Move mv = decode(p, a);
        bool win = p.height(mv.f) == 2 && p.height(mv.t) == 3;
        Pos c = win ? p : makeChild(p, mv);
        return py::make_tuple(fromPositions({c}), win);
    }, "Child state (opponent to move) after action a, and whether a is a winning step-up.");

    m.def("label", [](arr<int8_t> states, uint64_t rootBudget, uint64_t childBudget, int threads,
                      double ttGb) {
        auto ps = toPositions(states);
        std::vector<LabelOut> out;
        {
            py::gil_scoped_release nogil;
            labelBatch(ps, rootBudget, childBudget, threads, ttGb, out);
        }
        py::ssize_t n = ps.size();
        arr<int8_t> v(n);
        arr<uint8_t> w({n, (py::ssize_t)NMOVE});
        for (py::ssize_t i = 0; i < n; i++) {
            v.mutable_data()[i] = out[i].value;
            memcpy(w.mutable_data() + i * NMOVE, out[i].win, NMOVE);
        }
        return py::make_tuple(v, w);
    }, py::arg("states"), py::arg("root_budget") = 200000, py::arg("child_budget") = 20000,
       py::arg("threads") = 4, py::arg("tt_gb") = 0.25,
       "Exact values (+1/-1/0 unresolved) and proven winning actions.");

    m.def("generate", [](int n, int minB, int maxB, uint64_t rootBudget, uint64_t childBudget,
                         int threads, double ttGb, uint64_t seed) {
        std::vector<Pos> ps;
        std::vector<LabelOut> out;
        {
            py::gil_scoped_release nogil;
            generateLabeled(n, minB, maxB, rootBudget, childBudget, threads, ttGb, seed, ps, out);
        }
        arr<int8_t> v(n);
        arr<uint8_t> w({(py::ssize_t)n, (py::ssize_t)NMOVE});
        for (int i = 0; i < n; i++) {
            v.mutable_data()[i] = out[i].value;
            memcpy(w.mutable_data() + (size_t)i * NMOVE, out[i].win, NMOVE);
        }
        return py::make_tuple(fromPositions(ps), v, w);
    }, py::arg("n"), py::arg("min_blocks"), py::arg("max_blocks"), py::arg("root_budget") = 100000,
       py::arg("child_budget") = 5000, py::arg("threads") = 4, py::arg("tt_gb") = 0.25,
       py::arg("seed") = 1,
       "n solver-labelled positions from random safe play, blocks uniform in [min, max].");

    m.def("pretty", [](arr<int8_t> state) {
        Pos p = toPositions(state.reshape({1, STATE_LEN}))[0];
        std::string s = "    A  B  C  D  E\n";
        for (int r = 0; r < 5; r++) {
            s += " " + std::to_string(r + 1) + " ";
            for (int c = 0; c < 5; c++) {
                int i = r * 5 + c, h = p.height(i);
                char w = (p.me >> i & 1) ? 'X' : (p.op >> i & 1) ? 'O' : ' ';
                s += h == 4 ? " []" : " " + std::to_string(h) + w;
            }
            s += "\n";
        }
        return s + "(X = to move, O = opponent, [] = dome)";
    });

    m.def("all_moves", [](arr<int8_t> state) {
        Pos p = toPositions(state.reshape({1, STATE_LEN}))[0];
        auto ms = allMoves(p);
        arr<int32_t> out({(py::ssize_t)ms.size(), (py::ssize_t)3});
        for (size_t i = 0; i < ms.size(); i++) {
            out.mutable_data()[i * 3] = ms[i].f;
            out.mutable_data()[i * 3 + 1] = ms[i].t;
            out.mutable_data()[i * 3 + 2] = ms[i].b == 255 ? -1 : ms[i].b;
        }
        return out;
    }, "Every legal (from, to, build) for the side to move, unpruned; build = -1 for a step-up win.");

    m.def("action_index", [](arr<int8_t> state, int f, int t, int b) {
        Pos p = toPositions(state.reshape({1, STATE_LEN}))[0];
        return encodeMove(p, Move{(uint8_t)f, (uint8_t)t, (uint8_t)(b < 0 ? f : b)});
    }, "Policy index of (from, to, build); build = -1 for a step-up win.");

    py::class_<SearchParams>(m, "SearchParams")
        .def(py::init<>())
        .def_readwrite("cpuct", &SearchParams::cpuct)
        .def_readwrite("root_budget", &SearchParams::rootBudget)
        .def_readwrite("child_budget", &SearchParams::childBudget)
        .def_readwrite("tt_gb", &SearchParams::ttGb);

    auto moveDict = [](const MoveStat& s) {
        py::dict d;
        d["from"] = (int)s.m.f; d["to"] = (int)s.m.t; d["build"] = s.stepUp ? -1 : (int)s.m.b;
        d["action"] = s.a; d["step_up"] = s.stepUp; d["visits"] = s.N; d["q"] = s.Q;
        d["prior"] = s.P; d["proof"] = s.proof;
        py::list pv;
        for (size_t i = 0; i < s.pv.size(); i++) {
            bool win = s.pvEndsInWin && i + 1 == s.pv.size();
            pv.append(py::make_tuple((int)s.pv[i].f, (int)s.pv[i].t, win ? -1 : (int)s.pv[i].b));
        }
        d["pv"] = pv;
        d["pv_ends_in_win"] = s.pvEndsInWin;
        return d;
    };

    py::class_<Search>(m, "Search")
        .def(py::init([](arr<int8_t> state, const SearchParams& sp) {
            Pos p = toPositions(state.reshape({1, STATE_LEN}))[0];
            py::gil_scoped_release nogil;
            return new Search(p, sp);
        }), "Search from a state (side to move's perspective). Runs the root solve if enabled.")
        .def("prepare", [](Search& s, arr<float> planes, int maxLeaves) {
            if (planes.ndim() != 4 || planes.shape(0) < maxLeaves)
                throw std::runtime_error("planes buffer must be [>=max_leaves, 5, 5, 15]");
            float* P = planes.mutable_data();
            py::gil_scoped_release nogil;
            return s.prepare(P, maxLeaves);
        }, "Collect up to max_leaves leaves for the network; returns how many.")
        .def("apply", [](Search& s, arr<float> logits, arr<float> values) {
            s.apply(logits.data(), values.data());
        })
        .def_property_readonly("done", &Search::done)
        .def_property_readonly("simulations", &Search::simulations)
        .def_property_readonly("status", &Search::status)
        .def_property_readonly("solver_result", &Search::solverResult)
        .def_property_readonly("net_value", &Search::netValue)
        .def_property_readonly("search_value", &Search::searchValue)
        .def_property_readonly("tree_size", &Search::treeSize)
        .def("moves", [moveDict](Search& s, int pvDepth) {
            py::list out;
            for (auto& st : s.stats(pvDepth)) out.append(moveDict(st));
            return out;
        }, py::arg("pv_depth") = 8, "Per-move statistics, most visited first.")
        .def("best", [moveDict](Search& s) { return moveDict(s.best()); })
        .def("step_select", [](Search& s, arr<float> planes) {
            if (planes.ndim() != 4 || planes.shape(0) < 1) throw std::runtime_error("planes must be [>=1, 5, 5, 15]");
            return s.stepSelect(planes.mutable_data());
        }, "1 = leaf needs the network (planes[0] written), 2 = value already known, 0 = finished")
        .def("step_expand", [](Search& s, arr<float> logits, float value) { s.stepExpand(logits.data(), value); })
        .def("step_expand_known", &Search::stepExpandKnown)
        .def("step_backup", &Search::stepBackup)
        .def_property_readonly("step_phase", &Search::stepPhase)
        .def_property_readonly("step_leaf", &Search::stepLeafNode)
        .def_property_readonly("step_value", &Search::stepLeafValue)
        .def_property_readonly("step_nodes", &Search::stepNodes)
        .def_property_readonly("node_count", &Search::nodeCount)
        .def("tree", [](Search& s, int root, int maxNodes, float prunePct, int maxDepth, bool unvisited) {
            auto items = s.exportTree(root, maxNodes, prunePct, maxDepth, unvisited);
            py::list out;
            for (auto& t : items) {
                py::dict d;
                d["id"] = t.id; d["parent"] = t.parent == INT_MIN ? py::object(py::none()) : py::object(py::int_(t.parent));
                d["depth"] = t.depth; d["n"] = t.N; d["q"] = t.Q; d["p"] = t.P; d["u"] = t.U;
                d["status"] = t.status; d["expanded"] = t.expanded;
                d["net"] = std::isnan(t.netV) ? py::object(py::none()) : py::object(py::float_(t.netV));
                d["move"] = t.hasMove ? py::object(py::make_tuple((int)t.m.f, (int)t.m.t, (int)t.m.b)) : py::object(py::none());
                py::list h;
                for (int c = 0; c < 25; c++) h.append(t.p.height(c));
                d["heights"] = h;
                d["me"] = py::make_tuple(lsb(t.p.me), lsb(t.p.me & (t.p.me - 1)));
                d["op"] = py::make_tuple(lsb(t.p.op), lsb(t.p.op & (t.p.op - 1)));
                d["pv"] = t.onPV; d["step"] = t.onStep; d["moves"] = t.childCount;
                out.append(d);
            }
            return out;
        }, py::arg("root") = 0, py::arg("max_nodes") = 150, py::arg("prune_pct") = 100.0f,
           py::arg("max_depth") = 8, py::arg("unvisited") = false,
           "Nodes of the search tree for display, most visited first, parents before children.");

    py::class_<Config>(m, "Config")
        .def(py::init<>())
        .def_readwrite("n_games", &Config::nGames)
        .def_readwrite("sims", &Config::sims)
        .def_readwrite("cpuct", &Config::cpuct)
        .def_readwrite("dir_alpha", &Config::dirAlpha)
        .def_readwrite("dir_frac", &Config::dirFrac)
        .def_readwrite("noise", &Config::noise)
        .def_readwrite("temp_plies", &Config::tempPlies)
        .def_readwrite("label_min_blocks", &Config::labelMinBlocks)
        .def_readwrite("root_budget", &Config::rootBudget)
        .def_readwrite("child_budget", &Config::childBudget)
        .def_readwrite("threads", &Config::threads)
        .def_readwrite("tt_gb", &Config::ttGb)
        .def_readwrite("max_games", &Config::maxGames)
        .def_readwrite("arena", &Config::arena)
        .def_readwrite("seed", &Config::seed);

    py::class_<SelfPlay>(m, "SelfPlay")
        .def(py::init<const Config&>())
        .def("prepare", [](SelfPlay& sp, arr<float> planes) {
            if (planes.ndim() != 4 || planes.shape(0) < sp.cfg.nGames)
                throw std::runtime_error("planes buffer must be [n_games, 5, 5, 15]");
            std::vector<int32_t> gi(sp.cfg.nGames), ni(sp.cfg.nGames);
            int cnt;
            float* P = planes.mutable_data();
            {
                py::gil_scoped_release nogil;
                cnt = sp.prepare(P, gi.data(), ni.data());
            }
            gi.resize(cnt);
            ni.resize(cnt);
            return py::make_tuple(cnt, arr<int32_t>(cnt, gi.data()), arr<int32_t>(cnt, ni.data()));
        }, "Advance games; writes leaf planes into the buffer. Returns (count, game_idx, net_id).")
        .def("apply", [](SelfPlay& sp, arr<int32_t> gameIdx, arr<float> logits, arr<float> values) {
            int cnt = (int)gameIdx.size();
            if (logits.ndim() != 2 || logits.shape(0) < cnt || logits.shape(1) != NACT || values.size() < cnt)
                throw std::runtime_error("logits must be [>=count, 153], values [>=count]");
            py::gil_scoped_release nogil;
            sp.apply(cnt, gameIdx.data(), logits.data(), values.data());
        })
        .def("harvest", [](SelfPlay& sp) {
            auto smp = sp.harvest();
            py::ssize_t n = smp.size();
            arr<int8_t> st({n, (py::ssize_t)STATE_LEN});
            arr<float> pi({n, (py::ssize_t)NMOVE});
            arr<float> q(n), z(n);
            arr<int8_t> ex(n), v(n), pl(n);
            for (py::ssize_t i = 0; i < n; i++) {
                memcpy(st.mutable_data() + i * STATE_LEN, smp[i].state, STATE_LEN);
                memcpy(pi.mutable_data() + i * NMOVE, smp[i].pi, sizeof(float) * NMOVE);
                q.mutable_data()[i] = smp[i].q;
                z.mutable_data()[i] = smp[i].z;
                ex.mutable_data()[i] = smp[i].exact;
                v.mutable_data()[i] = smp[i].v;
                pl.mutable_data()[i] = smp[i].player;
            }
            py::dict d;
            d["state"] = st; d["pi"] = pi; d["q"] = q; d["z"] = z;
            d["exact"] = ex; d["v"] = v; d["player"] = pl;
            return d;
        }, "Samples from finished games since the last call.")
        .def_property_readonly("games_started", &SelfPlay::gamesStarted)
        .def_property_readonly("games_finished", &SelfPlay::gamesFinished)
        .def_property_readonly("active_games", &SelfPlay::activeGames)
        .def_property_readonly("net_wins", [](SelfPlay& sp) { return py::make_tuple(sp.netWins[0], sp.netWins[1]); })
        .def_property_readonly("first_player_wins", [](SelfPlay& sp) { return sp.firstPlayerWins; })
        .def_property_readonly("game_lengths", [](SelfPlay& sp) { return sp.gameLengths; })
        .def_property_readonly("stats", [](SelfPlay& sp) {
            py::dict d;
            d["games_started"] = sp.gamesStarted(); d["games_finished"] = sp.gamesFinished();
            d["active_games"] = sp.activeGames(); d["pending_samples"] = sp.pendingSamples();
            d["moves_played"] = sp.movesPlayed; d["root_solves"] = sp.rootSolves;
            d["root_proven"] = sp.rootProven; d["solver_moves"] = sp.solverMoves;
            d["leaf_evals"] = sp.leafEvals;
            return d;
        }, "Counters: moves played, exact root solves (and how many were proven), moves played "
           "straight from a proof, network leaf evaluations.");
}

Overwriting santorini_py.cpp


In [18]:
%%writefile santorini_agent.py
"""Santorini agent: the trained network + MCTS + exact solver, a game referee, and
the API behind the play widget (play.html).

    python santorini_agent.py --ckpt latest.pkl [--port 8000]     # local web app
    # in Colab / Jupyter:
    from santorini_agent import Net, Session, show_colab
    show_colab(Session(Net("latest.pkl")))

Cells are 0..24 = row*5 + col, written A1..E5 (column letter, row number, row 1 on top).
Player 0 ("Blue") places first and moves first; player 1 is "Gold".
"""
import argparse
import json
import os
import pickle
import time

import numpy as np

import santorini as S

NAMES = ["Blue", "Gold"]
DR = [-1, -1, -1, 0, 0, 1, 1, 1]
DC = [-1, 0, 1, -1, 1, -1, 0, 1]


def cell_name(c):
    return "ABCDE"[c % 5] + str(c // 5 + 1)


def move_text(f, t, b):
    return f"{cell_name(f)}–{cell_name(t)} wins" if b < 0 else f"{cell_name(f)}–{cell_name(t)} build {cell_name(b)}"


# --------------------------------------------------------------------------------------
# Network
# --------------------------------------------------------------------------------------
class Net:
    """The trained policy/value network. `ckpt` is the training notebook's latest.pkl,
    a weights_XXXX.msgpack / phase0_state.msgpack file, or None for a uniform network
    (zero logits, zero value: plain MCTS + solver, useful for testing)."""

    def __init__(self, ckpt=None):
        self.info = {"source": "uniform (no network)", "iteration": None, "params": 0}
        self.variables = None
        if ckpt is None:
            return
        import jax
        import jax.numpy as jnp
        import flax.linen as nn
        from flax import serialization

        raw_bytes, meta = None, {}
        if ckpt.endswith(".pkl"):
            ck = pickle.load(open(ckpt, "rb"))
            raw_bytes = ck["state"]
            meta = {"iteration": ck.get("iter"), "trained_on": ck.get("backend")}
            hist = [h for h in ck.get("history", []) if h.get("phase") == 1]
            if hist:
                meta["test_won_top1_winning"] = round(hist[-1].get("won_top1_winning", float("nan")), 3)
        else:
            raw_bytes = open(ckpt, "rb").read()
        raw = serialization.msgpack_restore(raw_bytes)
        params, stats = raw["params"], raw["batch_stats"]
        blocks = sum(1 for k in params if k.startswith("ResBlock_"))
        ch = int(np.asarray(params["Conv_0"]["kernel"]).shape[-1])

        class ResBlock(nn.Module):
            ch: int
            @nn.compact
            def __call__(self, x):
                y = nn.Conv(self.ch, (3, 3), use_bias=False)(x)
                y = nn.relu(nn.BatchNorm(use_running_average=True, momentum=0.9)(y))
                y = nn.Conv(self.ch, (3, 3), use_bias=False)(y)
                y = nn.BatchNorm(use_running_average=True, momentum=0.9)(y)
                return nn.relu(x + y)

        class SantoriniNet(nn.Module):
            blocks: int
            ch: int
            @nn.compact
            def __call__(self, x):
                bn = lambda: nn.BatchNorm(use_running_average=True, momentum=0.9)
                x = nn.relu(bn()(nn.Conv(self.ch, (3, 3), use_bias=False)(x)))
                for _ in range(self.blocks):
                    x = ResBlock(self.ch)(x)
                p = nn.relu(bn()(nn.Conv(32, (1, 1), use_bias=False)(x)))
                p = nn.Dense(S.NACT)(p.reshape(p.shape[0], -1))
                v = nn.relu(bn()(nn.Conv(16, (1, 1), use_bias=False)(x)))
                v = nn.relu(nn.Dense(128)(v.reshape(v.shape[0], -1)))
                return p, jnp.tanh(nn.Dense(1)(v))[:, 0]

        model = SantoriniNet(blocks, ch)
        self.variables = jax.device_put({"params": params, "batch_stats": stats})
        self._fwd = jax.jit(lambda var, x: model.apply(var, x))
        n_params = sum(np.asarray(x).size for x in jax.tree_util.tree_leaves(params))
        self.info = {"source": os.path.basename(ckpt), "blocks": blocks, "channels": ch,
                     "params": int(n_params), "device": jax.devices()[0].device_kind, **meta}

    def __call__(self, planes, n, pad_to=None):
        """Evaluate the first n rows of planes [*, 5, 5, 15] -> (logits [n, 153], values [n])."""
        if self.variables is None:
            return np.zeros((n, S.NACT), np.float32), np.zeros(n, np.float32)
        size = pad_to or n
        x = planes[:size] if len(planes) >= size else np.concatenate(
            [planes, np.zeros((size - len(planes),) + planes.shape[1:], planes.dtype)])
        if n < size:
            x = x.copy(); x[n:] = 0
        lg, v = self._fwd(self.variables, x)
        return np.asarray(lg, np.float32)[:n], np.asarray(v, np.float32)[:n]


# --------------------------------------------------------------------------------------
# Agent: search for one position at a time
# --------------------------------------------------------------------------------------
class Agent:
    def __init__(self, net, batch=16, cpuct=1.5, solver=True, root_budget=200_000, child_budget=2_000):
        self.net, self.batch = net, batch
        self.params = S.SearchParams()
        self.params.cpuct = cpuct
        self.set_solver(solver, root_budget, child_budget)
        self.buf = np.zeros((batch, 5, 5, S.NPLANES), np.float32)
        self.search, self.key, self.started, self.solve_seconds, self.think_seconds = None, None, 0, 0.0, 0.0

    def set_solver(self, on, root_budget=200_000, child_budget=2_000):
        self.params.root_budget = root_budget if on else 0
        self.params.child_budget = child_budget

    def begin(self, state, key):
        """Start (or keep) the search tree for this position."""
        if self.search is not None and self.key == key:
            return
        t = time.time()
        self.search = S.Search(np.asarray(state, np.int8), self.params)
        self.solve_seconds = time.time() - t
        self.key, self.think_seconds = key, 0.0

    def think(self, sims):
        """Run up to `sims` more simulations (stops early if the root is decided)."""
        s, t = self.search, time.time()
        target = s.simulations + sims
        while s.simulations < target and not s.done:
            n = s.prepare(self.buf, self.batch)
            if n == 0:
                if s.done:
                    break
                continue
            lg, v = self.net(self.buf, n, pad_to=self.batch)
            s.apply(np.ascontiguousarray(lg), np.ascontiguousarray(v))
        self.think_seconds += time.time() - t

    def snapshot(self, top=10, pv_depth=8):
        s = self.search
        moves = s.moves(pv_depth)
        total = max(1, sum(m["visits"] for m in moves))
        out = []
        for m in moves[:top]:
            out.append({"from": m["from"], "to": m["to"], "build": m["build"], "text": move_text(m["from"], m["to"], m["build"]),
                        "visits": m["visits"], "share": m["visits"] / total, "q": m["q"], "prior": m["prior"],
                        "proof": m["proof"], "step_up": m["step_up"],
                        "pv": [list(x) for x in m["pv"]], "pv_ends_in_win": m["pv_ends_in_win"]})
        best = s.best()
        return {"simulations": s.simulations, "status": s.status, "solver": s.solver_result,
                "net_value": s.net_value, "search_value": s.search_value if s.simulations else s.net_value,
                "done": s.done, "tree_size": s.tree_size, "n_moves": len(moves),
                "solve_seconds": round(self.solve_seconds, 2), "think_seconds": round(self.think_seconds, 2),
                "moves": out,
                "best": {"from": best["from"], "to": best["to"], "build": best["build"],
                         "text": move_text(best["from"], best["to"], best["build"])} if best["action"] >= 0 else None}


# --------------------------------------------------------------------------------------
# Opening placement (the network was trained on random placements, so the agent picks
# placements by evaluating the resulting positions with the value head).
# --------------------------------------------------------------------------------------
def _sym_cells():
    # D4 symmetries of the 5x5 board (same convention as the C++ tables).
    out = []
    for s in range(8):
        m = []
        for c in range(25):
            x, y = c % 5, c // 5
            for _ in range(s & 3):
                x, y = 4 - y, x
            if s >= 4:
                x = 4 - x
            m.append(y * 5 + x)
        out.append(m)
    return out


SYM = _sym_cells()


class Placement:
    """Incremental placement search so the widget can show progress.
    first=True : the agent places first (maximise over its pair, minimise over the reply).
    first=False: the agent places second against fixed opponent workers."""

    def __init__(self, net, first, opp=None, rng=None):
        self.net, self.first, self.opp = net, first, opp or []
        self.rng = rng or np.random.default_rng()
        free = [c for c in range(25) if c not in self.opp]
        pairs = [(a, b) for i, a in enumerate(free) for b in free[i + 1:]]
        if first:  # one representative per symmetry class
            seen, reps = set(), []
            for p in pairs:
                key = min(tuple(sorted((SYM[s][p[0]], SYM[s][p[1]]))) for s in range(8))
                if key not in seen:
                    seen.add(key); reps.append(p)
            self.cands = reps
        else:
            self.cands = pairs
        self.scores = {}       # pair -> score for the agent (-1..1)
        self.i = 0

    def _eval(self, me_pairs, op_pairs):
        """Values for the side to move (me) in the positions right after placement."""
        st = np.zeros((len(me_pairs), 29), np.int8)
        st[:, 25:27] = np.sort(np.array(me_pairs), 1)
        st[:, 27:29] = np.sort(np.array(op_pairs), 1)
        x, _, _ = S.encode(st, np.zeros(len(st), np.int32))
        _, v = self.net(x, len(st), pad_to=256 * ((len(st) + 255) // 256))
        return v

    def step(self, budget=2048):
        """Evaluate about `budget` positions; returns True when finished."""
        done = 0
        while self.i < len(self.cands) and done < budget:
            if self.first:
                A = self.cands[self.i]
                free = [c for c in range(25) if c not in A]
                B = [(a, b) for i, a in enumerate(free) for b in free[i + 1:]]
                v = self._eval([A] * len(B), B)      # we (first player) move first
                self.scores[A] = float(v.min())      # the opponent picks the reply worst for us
                self.i += 1
                done += len(B)
            else:
                chunk = self.cands[self.i:self.i + 512]
                v = self._eval([tuple(self.opp)] * len(chunk), chunk)   # the opponent moves first
                for p, val in zip(chunk, v):
                    self.scores[p] = -float(val)
                self.i += len(chunk)
                done += len(chunk)
        return self.done

    @property
    def done(self):
        return self.i >= len(self.cands)

    def all_scores(self):
        """Scores for every concrete pair (symmetry classes expanded)."""
        if not self.first:
            return dict(self.scores)
        out = {}
        for p, v in self.scores.items():
            for s in range(8):
                out[tuple(sorted((SYM[s][p[0]], SYM[s][p[1]])))] = v
        return out

    def view(self, top=5):
        sc = self.all_scores()
        heat = [None] * 25
        for (a, b), v in sc.items():
            for c in (a, b):
                heat[c] = v if heat[c] is None else max(heat[c], v)
        ranked = sorted(sc.items(), key=lambda kv: -kv[1])[:top]
        return {"progress": self.i / max(1, len(self.cands)), "candidates": len(self.cands),
                "heat": heat, "top": [{"cells": list(p), "score": v} for p, v in ranked]}

    def choose(self, tolerance=0.02):
        sc = self.all_scores()
        best = max(sc.values())
        options = [p for p, v in sc.items() if v >= best - tolerance]
        return list(options[self.rng.integers(len(options))]), best


# --------------------------------------------------------------------------------------
# Game referee
# --------------------------------------------------------------------------------------
class Game:
    """Board, turn order, legality and history. frames[k] is the position after the
    first k log entries, so the timeline can show any earlier position."""

    def __init__(self, human=0):
        self.human, self.agent = human, 1 - human
        self.heights = [0] * 25
        self.workers = [[], []]
        self.phase, self.turn = "place", 0
        self.winner, self.reason = None, ""
        self.log, self.evals = [], []
        self.frames = [self._frame()]

    def _frame(self):
        return {"heights": list(self.heights), "workers": [list(self.workers[0]), list(self.workers[1])],
                "turn": self.turn, "phase": self.phase}

    def state(self, player=None):
        """29-byte state from `player`'s perspective (default: side to move)."""
        p = self.turn if player is None else player
        s = np.zeros(29, np.int8)
        s[:25] = self.heights
        s[25:27] = sorted(self.workers[p]); s[27:29] = sorted(self.workers[1 - p])
        return s

    def key(self):
        return (tuple(self.heights), tuple(sorted(self.workers[0])), tuple(sorted(self.workers[1])), self.turn)

    def legal(self):
        if self.phase != "play":
            return []
        return [tuple(int(v) for v in m) for m in S.all_moves(self.state())]

    def place(self, cell):
        assert self.phase == "place", "not in the placement phase"
        assert 0 <= cell < 25 and cell not in self.workers[0] + self.workers[1], "that cell is taken"
        self.workers[self.turn].append(int(cell))
        self.log.append({"player": self.turn, "kind": "place", "cell": int(cell),
                         "text": f"places on {cell_name(cell)}"})
        if len(self.workers[self.turn]) == 2:
            if self.turn == 0:
                self.turn = 1
            else:
                self.phase, self.turn = "play", 0
                self._check_stuck()
        self.frames.append(self._frame())

    def move(self, f, t, b):
        assert self.phase == "play", "the game is not in the move phase"
        if (f, t, b) not in set(self.legal()):
            raise ValueError(f"illegal move {move_text(f, t, b)}")
        p = self.turn
        self.workers[p][self.workers[p].index(f)] = t
        self.log.append({"player": p, "kind": "move", "from": f, "to": t, "build": b, "text": move_text(f, t, b)})
        if b < 0:
            self.phase, self.winner = "over", p
            self.reason = f"{NAMES[p]} stepped up to level 3 on {cell_name(t)}"
        else:
            self.heights[b] += 1
            self.turn = 1 - p
            self._check_stuck()
        self.frames.append(self._frame())

    def _check_stuck(self):
        if self.phase == "play" and not self.legal():
            self.phase, self.winner = "over", 1 - self.turn
            self.reason = f"{NAMES[self.turn]} has no legal move"

    def rewind(self, k):
        """Go back to the position after the first k log entries."""
        assert 0 <= k < len(self.frames), "no such position"
        f = self.frames[k]
        self.heights, self.workers = list(f["heights"]), [list(f["workers"][0]), list(f["workers"][1])]
        self.turn, self.phase = f["turn"], f["phase"]
        self.winner, self.reason = None, ""
        self.log, self.frames = self.log[:k], self.frames[:k + 1]
        self.evals = [e for e in self.evals if e["entry"] < k]

    def to_json(self):
        return {"phase": self.phase, "turn": self.turn, "human": self.human, "agent": self.agent,
                "winner": self.winner, "reason": self.reason, "heights": self.heights,
                "workers": self.workers, "log": self.log, "evals": self.evals, "frames": self.frames,
                "legal": [list(m) for m in self.legal()] if self.turn == self.human else [],
                "names": NAMES}


# --------------------------------------------------------------------------------------
# Session: the API used by the widget
# --------------------------------------------------------------------------------------
def _win(q):
    return None if q is None else (float(q) + 1) / 2


class Session:
    DEFAULTS = {"sims": 400, "solver": True}
    TREE_DEFAULTS = {"root": 0, "max_nodes": 120, "prune": 100, "depth": 6, "unvisited": False}

    def __init__(self, net, batch=None, seed=None, keep_thoughts=200):
        self.net = net
        self.settings = dict(self.DEFAULTS)
        if batch is None:
            batch = 16 if net.variables is not None else 1
        self.agent = Agent(net, batch=batch)
        self.rng = np.random.default_rng(seed)
        self.keep_thoughts = keep_thoughts
        self._reset(Game(human=0))

    def _reset(self, game):
        self.game, self.placement, self.last_thought = game, None, None
        self.thoughts = {}       # id -> {"search", "snapshot", "player", "entry", "kind"}
        self.next_id = 0
        self.live = None         # id of the search on the current position
        self.agent.search = None
        self.step_planes = np.zeros((1, 5, 5, S.NPLANES), np.float32)
        self.step_info = None

    # -- the live search -------------------------------------------------------------------
    def _live_search(self):
        g = self.game
        self.agent.begin(g.state(), g.key())
        if self.live is None or self.thoughts.get(self.live, {}).get("search") is not self.agent.search:
            self.live = self.next_id
            self.next_id += 1
            self.thoughts[self.live] = {"search": self.agent.search, "snapshot": None, "player": g.turn,
                                        "entry": None, "kind": "live"}
            # forget old searches
            old = sorted(k for k, t in self.thoughts.items() if t["entry"] is not None)
            for k in old[:-self.keep_thoughts]:
                del self.thoughts[k]
        return self.agent.search

    def _thought_json(self):
        g = self.game
        if g.phase != "play" or self.agent.search is None:
            return None
        snap = self.agent.snapshot()
        snap["player"] = g.turn
        snap["target"] = self.settings["sims"]
        snap["id"] = self.live
        return snap

    def _attach_thought(self, entry, kind):
        """Keep the live search as the thinking behind log entry `entry`."""
        t = self.thoughts.get(self.live)
        if t is None:
            return None
        snap = self._thought_json()
        if snap is None or snap["simulations"] == 0 and snap["status"] == 0:
            return None
        t.update({"snapshot": snap, "entry": entry, "kind": kind})
        self._record_eval(entry, snap, kind)
        return snap

    def _record_eval(self, entry, snap, kind):
        q = float(snap["status"]) if snap["status"] != 0 else snap["search_value"]
        blue = (q + 1) / 2 if snap["player"] == 0 else (1 - q) / 2
        self.game.evals = [e for e in self.game.evals if e["entry"] != entry] + [
            {"entry": entry, "blue": blue, "proven": snap["status"] != 0, "by": kind}]

    def _finish_step(self):
        s = self.agent.search
        if s is None:
            return
        if s.step_phase == 1:
            lg, v = self.net(self.step_planes, 1, pad_to=self.agent.batch)
            s.step_expand(np.ascontiguousarray(lg[0]), float(v[0]))
        elif s.step_phase == 2:
            s.step_expand_known()
        if s.step_phase == 3:
            s.step_backup()

    # -- tree ------------------------------------------------------------------------------
    def _tree_json(self, tid, root=0, max_nodes=120, prune=100, depth=6, unvisited=False):
        t = self.thoughts.get(tid)
        if t is None:
            return None
        search, p0 = t["search"], t["player"]
        items = search.tree(int(root), int(max_nodes), float(prune), int(depth), bool(unvisited))
        nodes = []
        for it in items:
            mover = p0 if it["depth"] % 2 == 0 else 1 - p0
            mv = it["move"]
            moved = None if mv is None else 1 - mover
            workers = [None, None]
            workers[mover], workers[1 - mover] = list(it["me"]), list(it["op"])
            unvisited_node = it["id"] < 0 or it["n"] == 0
            if mv is None:
                win, net_win = _win(it["q"]), _win(it["net"])
            else:
                win = None if unvisited_node else _win(it["q"])
                net_win = None if it["net"] is None else (1 - it["net"]) / 2
            proof = None
            if it["status"] == 1:
                proof = mover
            elif it["status"] == -1:
                proof = 1 - mover
            terminal = None
            if it["moves"] == 0 and it["status"] == 1:
                terminal = f"{NAMES[mover]} can step up and win"
            elif it["moves"] == 0 and it["status"] == -1:
                terminal = f"{NAMES[mover]} has no safe move"
            build = None if mv is None else mv[2]   # tree branches are never step-up wins (those are terminal nodes)
            nodes.append({
                "id": it["id"], "parent": it["parent"], "depth": it["depth"], "n": it["n"],
                "to_move": mover, "moved_by": moved, "win": win, "net": net_win,
                "prior": it["p"] if mv is not None else None, "q": it["q"], "u": it["u"],
                "puct": (it["q"] if not unvisited_node else 0.0) + it["u"] if mv is not None else None,
                "proof": proof, "terminal": terminal, "expanded": it["expanded"], "unvisited": unvisited_node,
                "move": None if mv is None else [mv[0], mv[1], build],
                "move_text": None if mv is None else move_text(mv[0], mv[1], build),
                "heights": it["heights"], "workers": workers, "pv": it["pv"], "step": it["step"],
                "moves": it["moves"]})
        return {"thought": tid, "root_player": p0, "nodes": nodes, "total_nodes": search.node_count,
                "simulations": search.simulations, "cpuct": float(self.agent.params.cpuct)}

    def _with_tree(self, out, tree):
        if tree:
            params = dict(self.TREE_DEFAULTS, **{k: v for k, v in tree.items() if k in self.TREE_DEFAULTS})
            target = tree.get("target", "live")
            tid = self.live if target == "live" else self._thought_for_entry(int(target))
            out["tree"] = self._tree_json(tid, **params) if tid is not None else None
        return out

    def _thought_for_entry(self, entry):
        for k, t in self.thoughts.items():
            if t["entry"] == entry:
                return k
        return None

    def _response(self, **extra):
        return {"game": self.game.to_json(), "settings": self.settings, "net": self.net.info,
                "step": self.step_info, **extra}

    # -- API ---------------------------------------------------------------------------------
    def handle(self, name, payload=None):
        payload = payload or {}
        try:
            fn = getattr(self, "api_" + name)
        except AttributeError:
            return {"error": f"unknown request '{name}'"}
        try:
            return fn(**payload)
        except (AssertionError, ValueError) as e:
            return {"error": str(e), **self._response()}

    def api_state(self, tree=None):
        return self._with_tree(self._response(thought=self.last_thought), tree)

    def api_new(self, human=0, sims=None, solver=None):
        if sims is not None:
            self.settings["sims"] = int(sims)
        if solver is not None:
            self.settings["solver"] = bool(solver)
        self.agent.set_solver(self.settings["solver"])
        self._reset(Game(human=int(human)))
        return self._response()

    def api_settings(self, sims=None, solver=None):
        if sims is not None:
            self.settings["sims"] = int(sims)
        if solver is not None and bool(solver) != self.settings["solver"]:
            self.settings["solver"] = bool(solver)
            self.agent.set_solver(self.settings["solver"])
            self.agent.search, self.live = None, None
        return self._response()

    def api_place(self, cell):
        g = self.game
        assert g.turn == g.human and g.phase == "place", "it is not your turn to place"
        g.place(int(cell))
        return self._response()

    def api_move(self, frm, to, build):
        g = self.game
        assert g.turn == g.human and g.phase == "play", "it is not your turn"
        entry = len(g.log)
        hint = self.agent.search is not None and self.agent.key == g.key() and self.agent.search.simulations > 0
        if (int(frm), int(to), int(build)) not in set(g.legal()):
            raise ValueError(f"illegal move {move_text(int(frm), int(to), int(build))}")
        if hint:
            self._finish_step()
            self._attach_thought(entry, "hint")
        g.move(int(frm), int(to), int(build))
        if hint:
            g.log[-1]["thought"] = self.live
        self.agent.search, self.live, self.last_thought, self.step_info = None, None, None, None
        return self._response()

    def api_think(self, sims=64, tree=None):
        """Think about the current position (the agent's turn, or a hint on yours)."""
        g = self.game
        if g.phase == "place" and g.turn == g.agent:
            if self.placement is None:
                self.placement = Placement(self.net, first=(g.agent == 0), opp=list(g.workers[g.human]), rng=self.rng)
            self.placement.step(budget=max(256, int(sims) * 16))
            return self._response(placement=self.placement.view())
        if g.phase != "play":
            return self._response()
        self._live_search()
        self._finish_step()
        self.agent.think(int(sims))
        self.last_thought = self._thought_json()
        return self._with_tree(self._response(thought=self.last_thought), tree)

    def api_step(self, action="one", n=1, tree=None):
        """Run the search one stage at a time: select, expand (network), backup."""
        g = self.game
        assert g.phase == "play", "stepping through the search works once the workers are placed"
        s = self._live_search()
        info = {}
        if action in ("select", "one"):
            self._finish_step()
            r = s.step_select(self.step_planes)
            if r == 0:
                info = {"phase": "done", "explain": "The search is finished: the result of this position is already proven."}
        if action in ("expand", "one") and s.step_phase in (1, 2):
            if s.step_phase == 1:
                lg, v = self.net(self.step_planes, 1, pad_to=self.agent.batch)
                s.step_expand(np.ascontiguousarray(lg[0]), float(v[0]))
            else:
                s.step_expand_known()
        if action in ("backup", "one") and s.step_phase == 3:
            s.step_backup()
        if action == "many":
            self._finish_step()
            self.agent.think(int(n))
        if not info:
            info = self._step_info(s)
        self.step_info = info
        self.last_thought = self._thought_json()
        return self._with_tree(self._response(thought=self.last_thought), tree)

    def _step_info(self, s):
        phase = {0: "idle", 1: "selected", 2: "selected", 3: "evaluated", 4: "backed_up"}[s.step_phase]
        if phase == "idle":
            return {"phase": "idle", "explain": ""}
        p0 = self.game.turn
        path_ids = list(s.step_nodes)
        leaf = s.step_leaf
        depth = len(path_ids) - 1
        leaf_mover = p0 if depth % 2 == 0 else 1 - p0
        items = {it["id"]: it for it in s.tree(0, 100000, 100.0, 64, False)} if depth else {}
        path = []
        for nid in path_ids[1:]:
            it = items.get(nid)
            if it and it["move"]:
                mv = it["move"]
                path.append({"id": nid, "text": move_text(mv[0], mv[1], mv[2]), "player": p0 if it["depth"] % 2 == 1 else 1 - p0})
        out = {"phase": phase, "path": path, "leaf": leaf, "leaf_player": leaf_mover, "known": s.step_phase == 2}
        who = NAMES[leaf_mover]
        moves = ", then ".join(f"{NAMES[x['player']]} {x['text']}" for x in path) or "no moves (the current position)"
        if phase == "selected":
            if s.step_phase == 2:
                res = "wins" if s.step_value > 0 else "loses"
                out["explain"] = (f"Select: starting from the current position, it took the move with the highest Q + U at each "
                                  f"step: {moves}. That position is already decided ({who} {res}), so no network call is needed.")
            elif not path:
                out["explain"] = ("Select: this is the first simulation, so there is nothing to choose between yet: the "
                                  "search starts at the current position itself, which the network has not evaluated.")
            else:
                out["explain"] = (f"Select: starting from the current position, it took the move with the highest Q + U "
                                  f"(average value so far plus an exploration bonus) at each step: {moves}. "
                                  f"The network has not looked at the resulting position yet.")
        elif phase == "evaluated":
            v = s.step_value
            sub = s.tree(leaf, 400, 100.0, 1, True)
            kids = sorted([x for x in sub if x["move"] is not None], key=lambda x: -x["p"])[:3]
            fav = ", ".join(f"{move_text(x['move'][0], x['move'][1], x['move'][2])} "
                            f"({100 * x['p']:.0f}%)" for x in kids)
            if out["known"]:
                out["explain"] = f"Evaluate: the result is known exactly, {who} {'wins' if v > 0 else 'loses'}, so the value is {v:+.0f}."
            else:
                out["explain"] = (f"Expand and evaluate: the network says {who}, to move there, has a {100 * (v + 1) / 2:.0f}% chance "
                                  f"to win (value {v:+.2f}). It adds that position's {sub[0]['moves'] if sub else 0} moves as new "
                                  f"branches with its priors; favourites: {fav}.")
            out["value"] = v
        else:
            out["explain"] = (f"Back up: every move on the path got one more visit, and its average value moved toward the "
                              f"evaluation ({s.step_value:+.2f} for {who}, flipped in sign at each step for the other player). "
                              f"The search has now run {s.simulations} simulations.")
        return out

    def api_tree(self, target="live", **params):
        tid = self.live if target == "live" else self._thought_for_entry(int(target))
        return {"tree": self._tree_json(tid, **{k: v for k, v in params.items() if k in self.TREE_DEFAULTS})
                if tid is not None else None}

    def api_thought(self, entry, tree=None):
        """The stored thinking behind log entry `entry` (for the timeline)."""
        tid = self._thought_for_entry(int(entry))
        snap = self.thoughts[tid]["snapshot"] if tid is not None else None
        e = self.game.log[int(entry)] if 0 <= int(entry) < len(self.game.log) else {}
        out = {"thought": snap, "placement": e.get("placement")}
        if tree:
            tree = dict(tree, target=int(entry))
        return self._with_tree(out, tree)

    def api_agent_play(self, tree=None):
        """Play the agent's move (finishing the thinking first if needed)."""
        g = self.game
        assert g.turn == g.agent and g.phase in ("place", "play"), "it is not the agent's turn"
        entry = len(g.log)
        if g.phase == "place":
            if self.placement is None:
                self.placement = Placement(self.net, first=(g.agent == 0), opp=list(g.workers[g.human]), rng=self.rng)
            while not self.placement.done:
                self.placement.step()
            view = self.placement.view()
            cells, score = self.placement.choose()
            for c in cells:
                g.place(c)
            g.log[-1]["note"] = f"expects {100 * (score + 1) / 2:.0f}% to win"
            g.log[-2]["placement"] = g.log[-1]["placement"] = view
            self.placement = None
            return self._response(placement=view, played=cells)
        self._live_search()
        self._finish_step()
        remaining = self.settings["sims"] - self.agent.search.simulations
        if remaining > 0:
            self.agent.think(remaining)
        snap = self._attach_thought(entry, "agent") or self._thought_json()
        best = snap["best"]
        if best is None:  # every legal move lets the opponent win at once: play any of them
            f, t, b = g.legal()[self.rng.integers(len(g.legal()))]
            best = {"from": f, "to": t, "build": b, "text": move_text(f, t, b)}
            snap["best"], snap["status"] = best, -1
            self._record_eval(entry, snap, "agent")
        tid = self.live
        g.move(best["from"], best["to"], best["build"])
        g.log[-1]["thought"] = tid
        g.log[-1]["note"] = (("proven win" if snap["status"] == 1 else "proven loss" if snap["status"] == -1 else
                              f"{100 * (snap['search_value'] + 1) / 2:.0f}% to win") + f", {snap['simulations']} sims")
        self.agent.search, self.live, self.step_info = None, None, None
        self.last_thought = None
        out = self._response(thought=snap, played=best)
        if tree:
            out = self._with_tree(out, dict(tree, target=entry))
        return out

    def api_rewind(self, entry):
        """Go back to the position before log entry `entry` (it must be your turn there)."""
        g = self.game
        k = int(entry)
        assert 0 <= k < len(g.frames), "no such position"
        f = g.frames[k]
        assert f["turn"] == g.human and f["phase"] != "over", "you can only continue from a position where it is your turn"
        g.rewind(k)
        for t in self.thoughts.values():
            if t["entry"] is not None and t["entry"] >= k:
                t["entry"] = None
        self.thoughts = {i: t for i, t in self.thoughts.items() if t["entry"] is not None}
        self.agent.search, self.live, self.placement, self.last_thought, self.step_info = None, None, None, None, None
        return self._response()

    def api_undo(self):
        """Take back your last move (and the agent's reply)."""
        g = self.game
        mine = [i for i, e in enumerate(g.log) if e["player"] == g.human]
        assert mine, "nothing to take back"
        return self.api_rewind(mine[-1])


# --------------------------------------------------------------------------------------
# Front ends
# --------------------------------------------------------------------------------------
def widget_html(transport):
    html = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "play.html")).read()
    return html.replace("/*TRANSPORT*/", f"window.SANTORINI_TRANSPORT = {json.dumps(transport)};")


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    return o


def show_colab(session):
    """Display the widget in a Colab output cell, talking to `session` in this kernel."""
    from IPython.display import HTML, JSON, display
    from google.colab import output

    def api(name, payload_json):
        return JSON(_clean(session.handle(name, json.loads(payload_json or "{}"))))

    output.register_callback("santorini_api", api)
    display(HTML(widget_html("colab")))


def show(session, port=8000):
    """Show the widget: in Colab through kernel callbacks, elsewhere (Jupyter) by starting
    the local web server in a background thread and embedding it."""
    try:
        import google.colab  # noqa: F401
        return show_colab(session)
    except ImportError:
        pass
    import threading
    from IPython.display import IFrame, display
    threading.Thread(target=serve, args=(session, port), daemon=True).start()
    time.sleep(0.5)
    print(f"Widget served at http://127.0.0.1:{port}/ (open it in a browser if the frame stays empty)")
    display(IFrame(f"http://127.0.0.1:{port}/", width="100%", height=1150))


def serve(session, port=8000, host="127.0.0.1"):
    from http.server import BaseHTTPRequestHandler, HTTPServer

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def _send(self, code, body, ctype):
            data = body.encode() if isinstance(body, str) else body
            self.send_response(code)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            if self.path in ("/", "/index.html"):
                self._send(200, widget_html("http"), "text/html; charset=utf-8")
            else:
                self._send(404, "not found", "text/plain")

        def do_POST(self):
            if not self.path.startswith("/api/"):
                return self._send(404, "not found", "text/plain")
            n = int(self.headers.get("Content-Length", 0))
            payload = json.loads(self.rfile.read(n) or b"{}")
            out = _clean(session.handle(self.path[5:], payload))
            self._send(200, json.dumps(out), "application/json")

    print(f"Santorini agent ({session.net.info.get('source')}) on http://{host}:{port}", flush=True)
    HTTPServer((host, port), Handler).serve_forever()


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default=None, help="latest.pkl or weights_XXXX.msgpack (default: no network)")
    ap.add_argument("--port", type=int, default=8000)
    ap.add_argument("--host", default="127.0.0.1")
    a = ap.parse_args()
    serve(Session(Net(a.ckpt)), a.port, a.host)

Overwriting santorini_agent.py


In [19]:
%%writefile play.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Santorini agent</title>
<style>
:root {
  --page: #e9eff2; --panel: #f8fafb; --ink: #15232d; --muted: #5d6d78; --line: #cfd9df;
  --turf: #b9c9a8; --turf-edge: #a3b591; --stone: #fbfbf7; --stone-edge: #a9b2a6; --dome: #23479b;
  --blue: #1f6fd1; --gold: #d99a12; --blue-tint: #dbe8f8; --gold-tint: #f8ecd0;
  --win: #1c8a4a; --loss: #c2372b; --focus: #1f6fd1; --step: #c2185b;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --page: #0e1a22; --panel: #15242e; --ink: #e6edf1; --muted: #93a6b2; --line: #2a3d4a;
    --turf: #4e6447; --turf-edge: #3f533a; --stone: #e9ece6; --stone-edge: #8d978a; --dome: #4a73d6;
    --blue: #4f97f2; --gold: #f0b43a; --blue-tint: #1c3550; --gold-tint: #3d3018;
    --win: #43c07a; --loss: #f06a5c; --step: #ff5c9a;
  }
}
* { box-sizing: border-box; }
html, body { margin: 0; background: var(--page); color: var(--ink);
  font: 14px/1.45 ui-sans-serif, system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
.app { max-width: 1180px; margin: 0 auto; padding: 16px; }
.top { display: flex; align-items: baseline; justify-content: space-between; gap: 12px; flex-wrap: wrap; }
h1 { font-size: 20px; margin: 0; font-weight: 650; letter-spacing: -0.01em; }
.net { color: var(--muted); font-size: 12.5px; }
.status { font-size: 17px; margin: 10px 0 10px; min-height: 26px; font-weight: 550; }
.who-0 { color: var(--blue); } .who-1 { color: var(--gold); }
.timeline { display: grid; grid-template-columns: auto 1fr auto; gap: 10px; align-items: center; margin-bottom: 14px;
  background: var(--panel); border: 1px solid var(--line); border-radius: 12px; padding: 8px 12px; }
.timeline input[type=range] { width: 100%; }
.timeline .tl-label { font-size: 13px; color: var(--muted); grid-column: 1 / -1; min-height: 18px; }
.timeline .tl-label b { color: var(--ink); font-weight: 600; }
.grid { display: grid; grid-template-columns: minmax(300px, 480px) minmax(320px, 1fr); gap: 18px; align-items: start; }
@media (max-width: 860px) { .grid, .row2 { grid-template-columns: 1fr !important; } }
.boardwrap { background: var(--panel); border: 1px solid var(--line); border-radius: 14px; padding: 10px; position: relative; }
.boardwrap .preview-tag { position: absolute; left: 14px; top: 12px; font-size: 12px; background: var(--ink); color: var(--page);
  padding: 2px 8px; border-radius: 6px; display: none; }
.boardwrap.previewing .preview-tag { display: block; }
svg.board { width: 100%; height: auto; display: block; touch-action: manipulation; }
.coords { fill: var(--muted); font-size: 11px; }
svg.board *:not(.turf) { pointer-events: none; }
svg.board .turf { pointer-events: all; }
.cell.click { cursor: pointer; }
.cell:focus-visible { outline: none; }
.cell:focus-visible .turf { stroke: var(--focus); stroke-width: 3; }
.panel { background: var(--panel); border: 1px solid var(--line); border-radius: 14px; padding: 14px 16px; }
.stack > .panel + .panel { margin-top: 14px; }
.panel h2 { font-size: 14px; margin: 0 0 8px; font-weight: 650; }
.panel h2 small { font-weight: 450; color: var(--muted); font-size: 12.5px; margin-left: 6px; }
.meter { position: relative; height: 26px; border-radius: 7px; overflow: hidden; display: flex; font-weight: 650; font-size: 12.5px; }
.meter .b { background: var(--blue); color: #fff; display: flex; align-items: center; padding-left: 8px; transition: width .35s ease; white-space: nowrap; }
.meter .g { background: var(--gold); color: #1b1300; display: flex; align-items: center; justify-content: flex-end; padding-right: 8px; flex: 1; white-space: nowrap; }
.meter .mid { position: absolute; left: 50%; top: 0; bottom: 0; width: 1px; background: rgba(255,255,255,.7); }
.facts { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin-top: 10px; }
.fact { font-size: 12px; color: var(--muted); }
.fact b { display: block; color: var(--ink); font-size: 15px; font-weight: 650; font-variant-numeric: tabular-nums; }
.proof { margin-top: 10px; font-size: 13px; padding: 7px 10px; border-radius: 8px; border: 1px solid var(--line); }
.proof.win { border-color: var(--win); color: var(--win); }
.proof.loss { border-color: var(--loss); color: var(--loss); }
.progress { height: 4px; background: var(--line); border-radius: 2px; margin-top: 10px; overflow: hidden; }
.progress i { display: block; height: 100%; width: 0; background: var(--ink); transition: width .2s linear; }
svg.chart { width: 100%; display: block; }
.legend { display: flex; flex-wrap: wrap; gap: 4px 16px; font-size: 12px; color: var(--muted); margin-top: 4px; }
.legend i { display: inline-block; width: 12px; height: 10px; margin-right: 5px; vertical-align: -1px; border-radius: 2px; }
.line { font-size: 12.5px; color: var(--muted); margin-top: 8px; min-height: 18px; }
.line span { color: var(--ink); }
details.tbl { margin-top: 6px; font-size: 12.5px; }
details.tbl summary { cursor: pointer; color: var(--muted); }
table.cands { width: 100%; border-collapse: collapse; font-size: 12.5px; font-variant-numeric: tabular-nums; margin-top: 6px; }
table.cands th { text-align: left; color: var(--muted); font-weight: 550; padding: 0 6px 4px 0; }
table.cands td { padding: 4px 6px 4px 0; border-top: 1px solid var(--line); }
.controls { display: flex; flex-wrap: wrap; gap: 8px 14px; align-items: center; margin-top: 12px; }
button, select { font: inherit; color: var(--ink); background: var(--panel); border: 1px solid var(--line);
  border-radius: 8px; padding: 5px 11px; cursor: pointer; }
button:hover:not(:disabled), select:hover { border-color: var(--muted); }
button.primary { background: var(--ink); color: var(--page); border-color: var(--ink); }
button.accent { border-color: var(--step); color: var(--step); font-weight: 600; }
button:disabled { opacity: .45; cursor: default; }
button:focus-visible, select:focus-visible, input:focus-visible { outline: 2px solid var(--focus); outline-offset: 2px; }
label.toggle { display: inline-flex; align-items: center; gap: 6px; cursor: pointer; user-select: none; }
label.rng { display: inline-flex; align-items: center; gap: 6px; font-size: 13px; }
label.rng output { font-variant-numeric: tabular-nums; min-width: 34px; color: var(--muted); }
label.rng input { width: 110px; }
.err { color: var(--loss); font-size: 13px; min-height: 18px; margin-top: 6px; }
.empty { color: var(--muted); font-size: 13px; }
.treepanel { margin-top: 18px; }
.treehead { display: flex; flex-wrap: wrap; align-items: baseline; justify-content: space-between; gap: 8px; }
.treebar { display: flex; flex-wrap: wrap; gap: 8px 16px; align-items: center; margin: 8px 0; font-size: 13px; }
.stepbar { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin: 4px 0 8px; padding: 8px 10px;
  border: 1px dashed var(--line); border-radius: 10px; }
.stepbar .lbl { font-size: 13px; font-weight: 600; margin-right: 4px; }
.explain { font-size: 13px; min-height: 20px; margin: 4px 0 8px; }
.explain b { color: var(--step); }
.crumb { font-size: 12.5px; color: var(--muted); }
.crumb button { padding: 1px 8px; font-size: 12px; margin-left: 6px; }
.treebox { position: relative; height: 560px; border: 1px solid var(--line); border-radius: 10px; overflow: hidden; background: var(--page); }
.treebox:fullscreen { height: 100vh; border-radius: 0; }
.treebox svg { width: 100%; height: 100%; display: block; cursor: grab; user-select: none; }
.treebox svg.dragging { cursor: grabbing; }
.treebox .zoombtns { position: absolute; right: 8px; top: 8px; display: flex; gap: 4px; }
.treebox .zoombtns button { padding: 2px 9px; background: var(--panel); }
.treebox .treemsg { position: absolute; inset: 0; display: grid; place-items: center; color: var(--muted); font-size: 13px; pointer-events: none; text-align: center; padding: 20px; }
g.tnode { cursor: pointer; }
g.tnode:hover rect.card { filter: brightness(0.97); }
.row2 { display: grid; grid-template-columns: 1.3fr 1fr; gap: 18px; margin-top: 18px; }
.log { max-height: 220px; overflow: auto; font-size: 12.5px; margin: 0; padding: 0; list-style: none; font-variant-numeric: tabular-nums; }
.log li { padding: 3px 4px; border-top: 1px solid var(--line); display: flex; gap: 8px; cursor: pointer; border-radius: 4px; }
.log li:first-child { border-top: 0; }
.log li.cur { background: color-mix(in srgb, var(--line) 60%, transparent); }
.log .n { color: var(--muted); width: 22px; text-align: right; flex: none; }
.log .note { color: var(--muted); margin-left: auto; text-align: right; }
.dot { width: 9px; height: 9px; border-radius: 50%; display: inline-block; margin-top: 5px; flex: none; }
@media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
</style>
</head>
<body>
<div class="app" id="app">
  <div class="top">
    <h1>Santorini against the agent</h1>
    <div class="net" id="net"></div>
  </div>
  <div class="status" id="status" aria-live="polite">Connecting to the agent…</div>

  <div class="timeline">
    <button id="tlPrev" aria-label="Previous move">Back</button>
    <input type="range" id="tl" min="0" max="0" value="0" aria-label="Game timeline">
    <button id="tlNext" aria-label="Next move">Forward</button>
    <div class="tl-label" id="tlLabel"></div>
  </div>

  <div class="grid">
    <div>
      <div class="boardwrap" id="boardwrap">
        <svg class="board" id="board" viewBox="-22 -22 544 544" role="grid" aria-label="Board"></svg>
        <div class="preview-tag" id="previewTag">Position from the tree</div>
      </div>
      <div class="err" id="err" role="alert"></div>
      <div class="controls">
        <button class="primary" id="newBlue">New game as Blue</button>
        <button id="newGold">New game as Gold</button>
        <button id="undo">Take back</button>
        <button id="playHere" hidden>Play from here</button>
        <button id="toNow" hidden>Back to the game</button>
      </div>
      <div class="controls">
        <label>Agent strength
          <select id="sims">
            <option value="64">Quick (64 simulations)</option>
            <option value="200">Casual (200)</option>
            <option value="400" selected>Normal (400)</option>
            <option value="1600">Strong (1,600)</option>
            <option value="4000">Deep (4,000)</option>
          </select>
        </label>
        <label class="toggle"><input type="checkbox" id="solver" checked> Exact solver</label>
        <label class="toggle"><input type="checkbox" id="hints"> Hints on my turn</label>
        <label class="toggle"><input type="checkbox" id="thoughts" checked> Arrows on the board</label>
      </div>
    </div>
    <div class="stack">
      <div class="panel">
        <h2 id="thinkTitle">What the agent thinks</h2>
        <div class="meter" aria-label="Winning chances">
          <div class="b" id="meterB" style="width:50%">Blue 50%</div><div class="g" id="meterG">50% Gold</div><div class="mid"></div>
        </div>
        <div class="facts">
          <div class="fact">Network's first impression<b id="fNet">–</b></div>
          <div class="fact">After search<b id="fSearch">–</b></div>
          <div class="fact">Simulations<b id="fSims">–</b></div>
        </div>
        <div class="proof" id="proof">Solver: no position yet</div>
        <div class="progress"><i id="prog"></i></div>
      </div>
      <div class="panel">
        <h2>Which moves did it try? <small id="chartSub"></small></h2>
        <svg class="chart" id="visits" viewBox="0 0 520 250" role="img" aria-label="Simulations per move"></svg>
        <div class="legend" id="visitsLegend"></div>
        <div class="line" id="line"></div>
        <details class="tbl"><summary>Show as a table</summary><div id="cands"></div></details>
      </div>
    </div>
  </div>

  <div class="panel treepanel">
    <div class="treehead">
      <h2>Search tree <small id="treeSub">Each box is a position the search reached; click one to explore from there</small></h2>
      <div class="crumb" id="crumb"></div>
    </div>
    <div class="treebar">
      <label class="rng">Max nodes <input type="range" id="tMax" min="10" max="600" step="10" value="120"><output id="tMaxV">120</output></label>
      <label class="rng" title="Keep the most-visited children that together cover this share of their parent's simulations">Children kept <input type="range" id="tPrune" min="5" max="100" step="5" value="100"><output id="tPruneV">100%</output></label>
      <label class="rng">Max depth <input type="range" id="tDepth" min="1" max="14" value="6"><output id="tDepthV">6</output></label>
      <label class="toggle"><input type="checkbox" id="tBoards" checked> Boards in nodes</label>
      <label class="toggle"><input type="checkbox" id="tPuct"> PUCT scores</label>
      <label class="toggle"><input type="checkbox" id="tMinimal"> Minimal text</label>
      <label class="toggle"><input type="checkbox" id="tHide" checked> Hide unvisited</label>
    </div>
    <div class="stepbar" id="stepbar">
      <span class="lbl">Step through the search:</span>
      <button id="sSelect">1. Select</button>
      <button id="sExpand">2. Expand and evaluate</button>
      <button id="sBackup">3. Back up</button>
      <button id="sOne">Run 1 simulation</button>
      <button id="sMany">Run 25</button>
      <label class="toggle" style="margin-left:auto"><input type="checkbox" id="pause"> Pause the agent on its turn</label>
      <button class="accent" id="letMove" hidden>Let the agent move</button>
    </div>
    <div class="explain" id="explain"></div>
    <div class="treebox" id="treebox">
      <svg id="tree" aria-label="Search tree"><g id="treeG"></g></svg>
      <div class="zoombtns">
        <button id="zOut" aria-label="Zoom out">−</button><button id="zIn" aria-label="Zoom in">+</button>
        <button id="zFit">Fit</button><button id="zFull">Full screen</button>
      </div>
      <div class="treemsg" id="treeMsg">The tree appears once there is a search to show.</div>
    </div>
    <div class="legend" style="margin-top:8px">
      <span><i style="background:var(--blue-tint);border:1.5px solid var(--blue)"></i>Blue just moved</span>
      <span><i style="background:var(--gold-tint);border:1.5px solid var(--gold)"></i>Gold just moved</span>
      <span><i style="border:2.5px solid var(--ink)"></i>Most-visited line</span>
      <span><i style="border:3px solid var(--step)"></i>Current simulation</span>
      <span>N = simulations through the node, % = win chance for the player who just moved, net = network's first impression</span>
    </div>
  </div>

  <div class="row2">
    <div class="panel">
      <h2>Winning chances over the game <small>agent's estimate for Blue; dark dots are solver-proven; click a dot to review that move</small></h2>
      <svg class="chart" id="eval" viewBox="0 0 520 170"></svg>
    </div>
    <div class="panel">
      <h2>Moves <small>click one to review it</small></h2>
      <ul class="log" id="log"></ul>
    </div>
  </div>
</div>
<script>
/*TRANSPORT*/
const byId = id => document.getElementById(id);
const NS = "http://www.w3.org/2000/svg";
const CELL = 100, NAMES = ["Blue", "Gold"];
const cellName = c => "ABCDE"[c % 5] + (Math.floor(c / 5) + 1);
const cx = c => (c % 5) * CELL + CELL / 2, cy = c => Math.floor(c / 5) * CELL + CELL / 2;
const pct = q => Math.round(50 * (q + 1));
const css = v => getComputedStyle(document.documentElement).getPropertyValue(v).trim();
const playerColor = p => css(p === 0 ? "--blue" : "--gold");
const tint = p => p === 0 ? css("--blue-tint") : p === 1 ? css("--gold-tint") : css("--panel");
const moveName = mv => mv[2] < 0 ? `${cellName(mv[0])}–${cellName(mv[1])} wins` : `${cellName(mv[0])}–${cellName(mv[1])} build ${cellName(mv[2])}`;

let G = null, net = {};
let liveThought = null, livePlacement = null, liveTree = null;
let view = null;                 // null = live game, else index of the log entry being reviewed
let past = { thought: null, placement: null, tree: null };
let busy = false, sel = null, hover = null, previewNode = null;
let hintToken = 0, agentToken = 0, chunk = 32;
let stepInfo = null;
const treeParams = { root: 0, max_nodes: 120, prune: 100, depth: 6, unvisited: false };

// ---------- transport ------------------------------------------------------------------
async function api(name, payload = {}) {
  let r;
  if (window.SANTORINI_TRANSPORT === "colab") {
    const res = await google.colab.kernel.invokeFunction("santorini_api", [name, JSON.stringify(payload)], {});
    r = res.data["application/json"];
  } else {
    const res = await fetch("/api/" + name, { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(payload) });
    r = await res.json();
  }
  byId("err").textContent = r.error || "";
  if (r.game) G = r.game;
  if (r.net) net = r.net;
  if ("step" in r) stepInfo = r.step;
  return r;
}
const treeReq = target => ({ target, ...treeParams });

// what is on display: the live game or a reviewed move
function shown() {
  if (view === null) return { thought: liveThought, placement: livePlacement, tree: liveTree, frame: null, entry: null };
  return { thought: past.thought, placement: past.placement, tree: past.tree, frame: G.frames[view], entry: G.log[view] };
}
function boardState() {
  if (previewNode) return { heights: previewNode.heights, workers: previewNode.workers };
  if (view !== null) return G.frames[view];
  return G;
}

// ---------- svg helpers -------------------------------------------------------------------
function el(tag, attrs = {}, parent) {
  const e = document.createElementNS(NS, tag);
  for (const [k, v] of Object.entries(attrs)) e.setAttribute(k, v);
  if (parent) parent.appendChild(e);
  return e;
}
function txt(parent, x, y, s, attrs = {}) { const t = el("text", { x, y, ...attrs }, parent); t.textContent = s; return t; }

// ---------- main board ------------------------------------------------------------------------
function legalFrom(f) { return G.legal.filter(m => m[0] === f); }
function humanToMove() { return G && view === null && G.phase !== "over" && G.turn === G.human && !busy; }

function drawBoard() {
  const svg = byId("board");
  svg.innerHTML = "";
  byId("boardwrap").classList.toggle("previewing", !!previewNode);
  const defs = el("defs", {}, svg);
  for (const [id, color] of [["ah0", playerColor(0)], ["ah1", playerColor(1)]]) {
    const m = el("marker", { id, viewBox: "0 0 10 10", refX: 7, refY: 5, markerWidth: 4, markerHeight: 4, orient: "auto-start-reverse" }, defs);
    el("path", { d: "M0,0 L10,5 L0,10 z", fill: color }, m);
  }
  for (let i = 0; i < 5; i++) {
    txt(svg, i * CELL + CELL / 2, -8, "ABCDE"[i], { "text-anchor": "middle", class: "coords" });
    txt(svg, -12, i * CELL + CELL / 2 + 4, i + 1, { "text-anchor": "middle", class: "coords" });
  }
  if (!G) return;
  const B = boardState(), sh = shown();
  const placing = !previewNode && (view === null ? G.phase === "place" : sh.frame.phase === "place");
  const heat = placing && sh.placement ? sh.placement.heat : null;
  let hmin = 1, hmax = -1;
  if (heat) heat.forEach(v => { if (v !== null) { hmin = Math.min(hmin, v); hmax = Math.max(hmax, v); } });
  const occupied = [...B.workers[0], ...B.workers[1]];
  const targets = new Map();
  if (!previewNode && humanToMove()) {
    if (G.phase === "place") for (let c = 0; c < 25; c++) if (!occupied.includes(c)) targets.set(c, "place");
    if (G.phase === "play" && sel) {
      if (sel.to === undefined) legalFrom(sel.from).forEach(m => targets.set(m[1], m[2] < 0 ? "win" : "move"));
      else legalFrom(sel.from).filter(m => m[1] === sel.to).forEach(m => targets.set(m[2], "build"));
    }
  }
  for (let c = 0; c < 25; c++) {
    const x = (c % 5) * CELL, y = Math.floor(c / 5) * CELL, h = B.heights[c];
    const clickable = !previewNode && (targets.has(c) || (humanToMove() && G.phase === "play" && G.workers[G.human].includes(c) && legalFrom(c).length));
    const g = el("g", { class: "cell" + (clickable ? " click" : ""), "data-cell": c, tabindex: clickable ? 0 : -1, role: "gridcell",
      "aria-label": `${cellName(c)}, level ${h === 4 ? "3 with dome" : h}` }, svg);
    el("rect", { class: "turf", x: x + 2, y: y + 2, width: CELL - 4, height: CELL - 4, rx: 10, fill: css("--turf"), stroke: css("--turf-edge") }, g);
    if (heat && heat[c] !== null && hmax > hmin) {
      const who = view === null ? G.agent : sh.entry.player;
      el("rect", { x: x + 2, y: y + 2, width: CELL - 4, height: CELL - 4, rx: 10, fill: playerColor(who), opacity: (0.08 + 0.55 * (heat[c] - hmin) / (hmax - hmin)).toFixed(2) }, g);
    }
    for (let l = 1; l <= Math.min(h, 3); l++) {
      const inset = 6 + (l - 1) * 11;
      el("rect", { x: x + inset, y: y + inset, width: CELL - 2 * inset, height: CELL - 2 * inset, rx: 9 - l, fill: css("--stone"), stroke: css("--stone-edge"), "stroke-width": 1.5 }, g);
    }
    if (h === 4) el("circle", { cx: x + 50, cy: y + 50, r: 21, fill: css("--dome"), stroke: "#0d2357", "stroke-width": 1.5 }, g);
    if (h > 0 && h < 4) txt(g, x + 12, y + 22, h, { "font-size": 12, fill: css("--muted"), "font-weight": 600 });
    const t = targets.get(c);
    if (t === "move" || t === "place") el("circle", { cx: x + 50, cy: y + 50, r: 8, fill: playerColor(G.human), opacity: .55 }, g);
    if (t === "win") el("circle", { cx: x + 50, cy: y + 50, r: 16, fill: "none", stroke: css("--win"), "stroke-width": 4 }, g);
    if (t === "build") el("rect", { x: x + 30, y: y + 30, width: 40, height: 40, rx: 6, fill: "none", stroke: playerColor(G.human), "stroke-width": 3, "stroke-dasharray": "6 4" }, g);
    g.addEventListener("click", () => onCell(c));
    g.addEventListener("keydown", e => { if (e.key === "Enter" || e.key === " ") { e.preventDefault(); onCell(c); } });
  }
  if (previewNode && previewNode.move) {
    const [f, t, b] = previewNode.move, p = previewNode.moved_by;
    arrow(svg, f, t, playerColor(p), 6, .95, "ah" + p);
    if (b >= 0) el("rect", { x: cx(b) - 13, y: cy(b) - 13, width: 26, height: 26, rx: 4, fill: playerColor(p), opacity: .9 }, svg);
  } else if (!previewNode) {
    // last move before this position
    const lastIdx = (view === null ? G.log.length : view) - 1;
    const last = lastIdx >= 0 ? G.log[lastIdx] : null;
    if (last && last.kind === "move") {
      el("line", { x1: cx(last.from), y1: cy(last.from), x2: cx(last.to), y2: cy(last.to), stroke: playerColor(last.player), "stroke-width": 3, opacity: .35, "stroke-linecap": "round" }, svg);
      if (last.build >= 0) el("rect", { x: cx(last.build) - 7, y: cy(last.build) + 26, width: 14, height: 5, rx: 2, fill: playerColor(last.player), opacity: .7 }, svg);
    }
    const th = sh.thought;
    const showArrows = th && !placing && (view !== null || (G.phase === "play" && th.player === G.turn)) && (byId("thoughts").checked || th.player === G.human);
    if (showArrows) drawThoughts(svg, th);
    if (view !== null && sh.entry && sh.entry.kind === "move") {   // the move actually played
      const e = sh.entry;
      el("circle", { cx: cx(e.to), cy: cy(e.to), r: 27, fill: "none", stroke: css("--ink"), "stroke-width": 2.5, "stroke-dasharray": "5 4" }, svg);
    }
    if (placing && sh.placement && sh.placement.top.length) {
      const cells = view !== null && sh.entry ? [sh.entry.cell] : sh.placement.top[0].cells;
      for (const c of cells) el("rect", { x: (c % 5) * CELL + 4, y: Math.floor(c / 5) * CELL + 4, width: CELL - 8, height: CELL - 8, rx: 9, fill: "none", stroke: css("--ink"), "stroke-width": 3 }, svg);
    }
  }
  for (const p of [0, 1]) for (const c of B.workers[p]) {
    const g = el("g", {}, svg);
    if (!previewNode && sel && sel.from === c) el("circle", { cx: cx(c), cy: cy(c), r: 25, fill: "none", stroke: playerColor(p), "stroke-width": 3, opacity: .6 }, g);
    el("circle", { cx: cx(c), cy: cy(c) + 2, r: 17, fill: "rgba(0,0,0,.18)" }, g);
    el("circle", { cx: cx(c), cy: cy(c), r: 17, fill: playerColor(p), stroke: "#fff", "stroke-width": 3 }, g);
  }
  if (!previewNode && hover !== null && sh.thought && sh.thought.moves[hover]) drawLine(svg, sh.thought, sh.thought.moves[hover]);
}

function arrow(svg, f, t, color, width, opacity, marker) {
  const x1 = cx(f), y1 = cy(f), x2 = cx(t), y2 = cy(t), dx = x2 - x1, dy = y2 - y1, len = Math.hypot(dx, dy);
  el("line", { x1: x1 + dx / len * 20, y1: y1 + dy / len * 20, x2: x2 - dx / len * 14, y2: y2 - dy / len * 14,
    stroke: color, "stroke-width": width, opacity, "stroke-linecap": "round", "marker-end": `url(#${marker})` }, svg);
}

function drawThoughts(svg, th) {
  const color = playerColor(th.player), mk = "ah" + th.player;
  const groups = new Map();
  th.moves.slice(0, 10).forEach((m, i) => {
    const k = m.from + "-" + m.to;
    if (!groups.has(k)) groups.set(k, { from: m.from, to: m.to, share: 0, best: false, stepUp: false, moves: [] });
    const g = groups.get(k);
    g.share += m.share; g.moves.push({ m, i }); g.best = g.best || i === 0; g.stepUp = g.stepUp || m.step_up;
  });
  const gs = [...groups.values()].sort((a, b) => b.share - a.share).slice(0, 5);
  const maxShare = Math.max(...gs.map(g => g.share), 1e-9);
  gs.slice().reverse().forEach(g => {
    const s = g.stepUp ? 1 : g.share / maxShare, op = g.best ? 0.95 : 0.25 + 0.45 * s;
    arrow(svg, g.from, g.to, color, g.best ? 7 : 2 + 4 * s, op, mk);
    for (const { m, i } of g.moves) {
      if (m.build < 0) continue;
      const size = 12 + 14 * Math.sqrt(m.share / maxShare);
      el("line", { x1: cx(m.to), y1: cy(m.to), x2: cx(m.build), y2: cy(m.build), stroke: color, "stroke-width": 1.5, "stroke-dasharray": "3 4", opacity: op * .7 }, svg);
      el("rect", { x: cx(m.build) - size / 2, y: cy(m.build) - size / 2, width: size, height: size, rx: 4, fill: i === 0 ? color : "none", stroke: color, "stroke-width": 2.5, opacity: i === 0 ? 0.95 : op }, svg);
    }
  });
  gs.slice(0, 3).forEach(g => {
    const tx = (cx(g.from) + cx(g.to)) / 2, ty = (cy(g.from) + cy(g.to)) / 2 - 8;
    txt(svg, tx, ty, g.stepUp ? "win" : Math.round(100 * g.share) + "%", { "text-anchor": "middle", "font-size": 13, "font-weight": 700,
      fill: css("--ink"), stroke: css("--panel"), "stroke-width": 4, "paint-order": "stroke" });
  });
}

function drawLine(svg, th, m) {
  const seq = [[m.from, m.to, m.build], ...m.pv];
  let mover = th.player;
  seq.forEach((mv, k) => {
    const color = playerColor(mover);
    arrow(svg, mv[0], mv[1], color, 3, .9, "ah" + mover);
    el("circle", { cx: cx(mv[1]) + 18, cy: cy(mv[1]) - 18, r: 10, fill: color }, svg);
    txt(svg, cx(mv[1]) + 18, cy(mv[1]) - 14, k + 1, { "text-anchor": "middle", "font-size": 11, "font-weight": 700, fill: "#fff" });
    if (mv[2] >= 0) el("rect", { x: cx(mv[2]) - 8, y: cy(mv[2]) - 8, width: 16, height: 16, rx: 3, fill: "none", stroke: color, "stroke-width": 2, "stroke-dasharray": "3 2" }, svg);
    mover = 1 - mover;
  });
}

// ---------- thinking panel ----------------------------------------------------------------------
function drawThinking() {
  const sh = shown();
  const th = sh.thought && (view !== null || (G && G.phase === "play" && sh.thought.player === G.turn)) ? sh.thought : null;
  const placing = G && (view === null ? G.phase === "place" : sh.frame.phase === "place");
  let title = "What the agent thinks";
  if (G) {
    if (view !== null) title = th ? `Thinking behind move ${view + 1}: ${NAMES[th.player]}${th.player === G.human ? " (hint for you)" : " (the agent)"}` :
      placing && sh.placement ? `How ${NAMES[sh.entry.player]} chose where to place` : `Move ${view + 1}: no thinking recorded`;
    else if (placing) title = G.turn === G.agent ? `Choosing where ${NAMES[G.agent]} places` : "Place your two workers";
    else if (th) title = th.player === G.agent ? `What ${NAMES[th.player]} (the agent) thinks` : `Hint for you (${NAMES[th.player]})`;
  }
  byId("thinkTitle").textContent = title;
  let blue = 0.5;
  const pr = byId("proof");
  pr.className = "proof";
  if (th) {
    const q = th.status !== 0 ? th.status : th.search_value;
    blue = th.player === 0 ? (q + 1) / 2 : (1 - q) / 2;
    const mover = NAMES[th.player];
    byId("fNet").textContent = `${pct(th.net_value)}% for ${mover}`;
    byId("fSearch").textContent = th.simulations ? `${pct(th.search_value)}% for ${mover}` : "–";
    byId("fSims").textContent = th.simulations.toLocaleString();
    if (th.status === 1) { pr.textContent = `Solver: ${mover} has a forced win.`; pr.classList.add("win"); }
    else if (th.status === -1) { pr.textContent = `Solver: ${mover} loses against perfect play.`; pr.classList.add("loss"); }
    else pr.textContent = th.solver === 0 && byId("solver").checked ? "Solver: not decided within its budget in this position." : "Solver: off or not decided.";
    byId("prog").style.width = Math.min(100, 100 * th.simulations / Math.max(1, th.target)) + "%";
  } else if (placing && sh.placement) {
    const who = view === null ? G.agent : sh.entry.player;
    byId("fNet").textContent = "–";
    byId("fSearch").textContent = sh.placement.top.length ? `${pct(sh.placement.top[0].score)}% for ${NAMES[who]}` : "–";
    byId("fSims").textContent = sh.placement.candidates.toLocaleString() + " options";
    pr.textContent = "Shaded cells: how good each cell is for the placing player's workers, judged by the network.";
    byId("prog").style.width = 100 * sh.placement.progress + "%";
  } else {
    ["fNet", "fSearch", "fSims"].forEach(i => byId(i).textContent = "–");
    pr.textContent = G && G.phase === "over" && view === null ? "Game over." : "Waiting for a position to think about.";
    byId("prog").style.width = "0%";
  }
  if (G && G.phase === "over" && view === null) blue = G.winner === 0 ? 1 : 0;
  byId("meterB").style.width = (100 * blue).toFixed(1) + "%";
  byId("meterB").textContent = `Blue ${Math.round(100 * blue)}%`;
  byId("meterG").textContent = `${100 - Math.round(100 * blue)}% Gold`;
  drawVisits(th, placing ? sh.placement : null);
}

function drawVisits(th, placement) {
  const svg = byId("visits");
  svg.innerHTML = "";
  const W = 520, H = 250, L = 34, R = 8, T = 22, B = 62;
  byId("visitsLegend").innerHTML = "";
  byId("cands").innerHTML = "";
  if (!th || !th.moves.length) {
    byId("chartSub").textContent = "";
    txt(svg, W / 2, H / 2, placement ? "Placement is judged by the network directly; see the shaded board." : "Bars appear here while the agent thinks.",
      { "text-anchor": "middle", "font-size": 13, fill: css("--muted") });
    byId("line").textContent = "";
    return;
  }
  const ms = th.moves.slice(0, 10);
  byId("chartSub").textContent = `${th.simulations.toLocaleString()} simulations, ${th.n_moves} possible moves`;
  const ymax = Math.max(0.05, ...ms.map(m => Math.max(m.share, m.prior))) * 1.18;
  const y = v => T + (H - T - B) * (1 - v / ymax);
  for (let i = 0; i <= 4; i++) {
    const v = ymax * i / 4;
    el("line", { x1: L, x2: W - R, y1: y(v), y2: y(v), stroke: css("--line"), "stroke-width": 1 }, svg);
    txt(svg, L - 5, y(v) + 4, Math.round(100 * v) + "%", { "text-anchor": "end", "font-size": 10, fill: css("--muted") });
  }
  const slot = (W - L - R) / ms.length, bw = Math.min(30, slot * 0.42);
  const color = playerColor(th.player);
  ms.forEach((m, i) => {
    const x0 = L + slot * i + slot / 2;
    const g = el("g", { "data-i": i, style: "cursor:pointer" }, svg);
    el("rect", { x: L + slot * i, y: T - 6, width: slot, height: H - T - B + 6, fill: hover === i ? css("--line") : "transparent", opacity: .5 }, g);
    el("rect", { x: x0 - bw, y: y(m.share), width: bw, height: Math.max(0, y(0) - y(m.share)), fill: color, rx: 2 }, g);
    el("rect", { x: x0 + 1, y: y(m.prior), width: bw * 0.7, height: Math.max(0, y(0) - y(m.prior)), fill: "none", stroke: css("--ink"), "stroke-width": 1.3, "stroke-dasharray": "3 2", rx: 2 }, g);
    const lab = m.step_up ? "wins" : m.proof === 1 ? "proven" : m.proof === -1 ? "loses" : m.visits ? pct(m.q) + "%" : "";
    txt(g, x0 - bw / 2, y(m.share) - 5, lab, { "text-anchor": "middle", "font-size": 11, "font-weight": 700,
      fill: m.proof === 1 || m.step_up ? css("--win") : m.proof === -1 ? css("--loss") : css("--ink") });
    const short = m.build < 0 ? `${cellName(m.from)}–${cellName(m.to)}` : `${cellName(m.from)}–${cellName(m.to)} ${cellName(m.build)}`;
    txt(g, x0, H - B + 14, short, { "text-anchor": "end", "font-size": 10.5, fill: i === 0 ? css("--ink") : css("--muted"),
      "font-weight": i === 0 ? 700 : 400, transform: `rotate(-38 ${x0} ${H - B + 14})` });
    g.addEventListener("mouseenter", () => { hover = i; drawBoard(); drawLineText(); drawVisits(th, null); });
    g.addEventListener("mouseleave", () => { hover = null; drawBoard(); drawLineText(); drawVisits(th, null); });
  });
  byId("visitsLegend").innerHTML = `<span><i style="background:${color}"></i>share of simulations</span>
    <span><i style="border:1.3px dashed ${css("--ink")}"></i>network's prior (before search)</span>
    <span>number on a bar: win chance for ${NAMES[th.player]} after that move</span>`;
  const rows = th.moves.slice(0, 12).map((m, i) => `<tr><td>${m.text}</td><td>${m.visits}</td><td>${m.visits ? pct(m.q) + "%" : "–"}</td>
    <td>${(100 * m.prior).toFixed(1)}%</td><td>${m.step_up ? "wins now" : m.proof === 1 ? "forced win" : m.proof === -1 ? "loses" : ""}</td></tr>`).join("");
  byId("cands").innerHTML = `<table class="cands"><thead><tr><th>Move</th><th>Visits</th><th>Win chance</th><th>Prior</th><th>Solver</th></tr></thead><tbody>${rows}</tbody></table>`;
  drawLineText();
}

function drawLineText() {
  const th = shown().thought;
  if (!th || !th.moves.length) { byId("line").textContent = ""; return; }
  const i = hover === null ? 0 : hover, m = th.moves[i];
  const seq = [[m.from, m.to, m.build], ...m.pv];
  let mover = th.player;
  const parts = seq.map((mv, k) => { const s = `${k + 1}. ${NAMES[mover]} ${moveName(mv)}`; mover = 1 - mover; return s; });
  byId("line").innerHTML = (hover === null ? "Expected line: " : "Line after this move: ") + `<span>${parts.join(", ")}</span>` +
    (m.pv_ends_in_win ? "" : seq.length > 1 ? ", …" : "") + (hover === null ? " (hover a bar to show its line on the board)" : "");
}

// ---------- search tree -----------------------------------------------------------------------------
const tv = { x: 0, y: 0, k: 1, fitted: false, key: null };
let treeLayout = null;

function nodeSize(opts) {
  if (opts.minimal) return { w: 74, h: (opts.boards ? 66 : 4) + 36 };
  return { w: 140, h: 20 + (opts.boards ? 74 : 0) + 34 + (opts.puct ? 13 : 0) };
}

function drawTree() {
  const sh = shown(), tree = sh.tree, g = byId("treeG");
  g.innerHTML = "";
  const msg = byId("treeMsg");
  const liveOk = view === null && G && G.phase === "play";
  ["sSelect", "sExpand", "sBackup", "sOne", "sMany"].forEach(id => byId(id).disabled = !liveOk || stepping || (busy && !paused()));
  byId("sExpand").disabled ||= !(stepInfo && stepInfo.phase === "selected");
  byId("sBackup").disabled ||= !(stepInfo && stepInfo.phase === "evaluated");
  byId("stepbar").style.opacity = view === null ? 1 : .5;
  byId("explain").innerHTML = view === null && stepInfo && stepInfo.explain ? stepExplain() : "";
  crumb(tree);
  if (!tree || !tree.nodes.length) {
    msg.style.display = "grid";
    msg.textContent = view !== null ? "No search tree was recorded for this move." :
      G && G.phase === "place" ? "The tree starts once both players have placed their workers." : "The tree appears once there is a search to show.";
    treeLayout = null;
    return;
  }
  msg.style.display = "none";
  const opts = { boards: byId("tBoards").checked, puct: byId("tPuct").checked, minimal: byId("tMinimal").checked };
  const { w: NW, h: NH } = nodeSize(opts);
  const GAPX = 14, GAPY = 46;
  const byIdMap = new Map(tree.nodes.map(n => [n.id, n]));
  const kids = new Map();
  for (const n of tree.nodes) if (n.parent !== null && byIdMap.has(n.parent)) {
    if (!kids.has(n.parent)) kids.set(n.parent, []);
    kids.get(n.parent).push(n);
  }
  for (const [k, list] of kids) {      // most visited in the middle, so the main line runs straight down
    list.sort((a, b) => b.n - a.n || (b.prior || 0) - (a.prior || 0));
    const out = [];
    list.forEach((c, i) => { if (i % 2) out.push(c); else out.unshift(c); });
    kids.set(k, out);
  }
  const root = tree.nodes[0];
  const pos = new Map();
  let cursor = 0;
  (function place(n, d) {          // simple tidy layout: leaves left to right, parents centred over children
    const ch = kids.get(n.id) || [];
    if (!ch.length) { pos.set(n.id, { x: cursor + NW / 2, y: d * (NH + GAPY) }); cursor += NW + GAPX; return; }
    ch.forEach(c => place(c, d + 1));
    const xs = ch.map(c => pos.get(c.id).x);
    pos.set(n.id, { x: (Math.min(...xs) + Math.max(...xs)) / 2, y: d * (NH + GAPY) });
  })(root, 0);
  const width = Math.max(NW, cursor - GAPX), height = Math.max(...[...pos.values()].map(p => p.y)) + NH;
  treeLayout = { width, height, rootX: pos.get(root.id).x };
  // edges
  for (const n of tree.nodes) {
    if (!pos.has(n.parent)) continue;
    const a = pos.get(n.parent), b = pos.get(n.id), par = byIdMap.get(n.parent);
    const share = par.n ? n.n / par.n : 0;
    const onStep = n.step && par.step, onPV = n.pv && par.pv;
    const y1 = a.y + NH, y2 = b.y, my = (y1 + y2) / 2;
    el("path", { d: `M${a.x},${y1} C${a.x},${my} ${b.x},${my} ${b.x},${y2}`, fill: "none",
      stroke: onStep ? css("--step") : onPV ? css("--ink") : n.moved_by !== null ? playerColor(n.moved_by) : css("--muted"),
      "stroke-width": onStep ? 5 : onPV ? 3.2 : 1 + 4 * share, opacity: onStep || onPV ? 0.95 : n.unvisited ? 0.35 : 0.55 }, g);
  }
  for (const n of tree.nodes) drawNode(g, n, pos.get(n.id), NW, NH, opts, tree);
  const key = tree.thought + ":" + treeParams.root + ":" + JSON.stringify(opts);
  if (tv.key !== key) { tv.key = key; readableView(); } else applyView();
}

function drawNode(g, n, p, NW, NH, opts, tree) {
  const grp = el("g", { class: "tnode", transform: `translate(${p.x - NW / 2},${p.y})`, "data-id": n.id }, g);
  const who = n.moved_by;   // the player whose move led here (null at the root)
  const stroke = n.step ? css("--step") : n.pv ? css("--ink") : who !== null ? playerColor(who) : css("--muted");
  if (n.step) el("rect", { x: -5, y: -5, width: NW + 10, height: NH + 10, rx: 13, fill: "none", stroke: css("--step"), "stroke-width": 7, opacity: .22 }, grp);
  el("rect", { class: "card", width: NW, height: NH, rx: 9, fill: who !== null ? tint(who) : css("--panel"), stroke,
    "stroke-width": n.step ? 3.2 : n.pv ? 2.6 : 1.4, "stroke-dasharray": n.unvisited ? "4 3" : "none", opacity: n.unvisited ? 0.6 : 1 }, grp);
  if (n.proof !== null) {   // proven nodes get a coloured stripe on the left
    const good = who !== null ? n.proof === who : n.proof === n.to_move;
    el("rect", { x: 0, y: 6, width: 5, height: NH - 12, rx: 2.5, fill: good ? css("--win") : css("--loss") }, grp);
  }
  const center = { "text-anchor": "middle" };
  const winLabel = n.win === null ? "unvisited" : `${NAMES[who !== null ? who : n.to_move]} ${Math.round(100 * n.win)}%`;
  let y = 0;
  if (!opts.minimal) {
    const title = n.move ? n.move_text : `Now: ${NAMES[n.to_move]} to move`;
    txt(grp, NW / 2, 14, title.length > 21 ? title.replace(" build ", " b ") : title, { ...center, "font-size": 10.5, "font-weight": 650, fill: css("--ink") });
    y = 20;
  } else y = 5;
  if (opts.boards) {
    const bs = opts.minimal ? 12 : 14;
    miniBoard(grp, n, (NW - 5 * bs) / 2, y, bs);
    y += 5 * bs + 4;
  }
  const line = (s, fs, wgt, fill) => { y += fs + 3; txt(grp, NW / 2, y, s, { ...center, "font-size": fs, "font-weight": wgt, fill }); };
  if (opts.minimal) {
    line(`N ${n.n}`, 10.5, 650, css("--ink"));
    line(n.win === null ? "–" : `${Math.round(100 * n.win)}%`, 10.5, 650, who !== null ? playerColor(who) : css("--ink"));
  } else {
    line(`N ${n.n}   ${winLabel}`, 11, 650, css("--ink"));
    let second;
    if (n.terminal) second = n.terminal;
    else if (n.proof !== null) second = `proven: ${NAMES[n.proof]} wins`;
    else second = [n.net !== null ? `net ${Math.round(100 * n.net)}%` : (n.expanded ? "" : "not evaluated"),
                   n.prior !== null ? `prior ${(100 * n.prior).toFixed(n.prior < 0.1 ? 1 : 0)}%` : ""].filter(Boolean).join("   ");
    line(second, 10, 500, n.proof !== null && !n.terminal ? (n.proof === (who !== null ? who : n.to_move) ? css("--win") : css("--loss")) : css("--muted"));
    if (opts.puct) line(n.puct !== null ? `Q ${n.unvisited ? "0.00" : n.q.toFixed(2)} + U ${n.u.toFixed(2)} = ${n.puct.toFixed(2)}` : "root", 9.5, 500, css("--ink"));
    if (n.id >= 0) txt(grp, NW - 6, 9, "#" + n.id, { "text-anchor": "end", "font-size": 8, fill: css("--muted") });
  }
  const title = el("title", {}, grp);
  title.textContent = [n.move ? `${NAMES[who]} ${n.move_text}` : "Current position", `Simulations through here: ${n.n}`,
    n.win !== null ? `Win chance for ${NAMES[who !== null ? who : n.to_move]} after search: ${Math.round(100 * n.win)}%` : "Not visited yet",
    n.net !== null ? `Network's first impression: ${Math.round(100 * n.net)}%` : "", n.prior !== null ? `Network's prior for this move: ${(100 * n.prior).toFixed(1)}%` : "",
    n.puct !== null ? `PUCT score: Q ${n.q.toFixed(3)} + U ${n.u.toFixed(3)} = ${n.puct.toFixed(3)}` : "", n.terminal || "",
    n.expanded && n.id >= 0 ? "Click to explore the tree from here" : ""].filter(Boolean).join("\n");
  grp.addEventListener("mouseenter", () => { previewNode = n; drawBoard(); });
  grp.addEventListener("mouseleave", () => { previewNode = null; drawBoard(); });
  grp.addEventListener("click", () => {
    if (tv.moved) return;
    if (n.id >= 0 && n.expanded && n.id !== treeParams.root) { treeParams.root = n.id; refreshTree(); }
  });
}

function miniBoard(parent, n, x, y, s) {
  const g = el("g", { transform: `translate(${x},${y})` }, parent);
  const shade = [css("--turf"), "#e3e6df", "#f3f4ef", "#ffffff"];
  for (let c = 0; c < 25; c++) {
    const r = Math.floor(c / 5), k = c % 5, h = n.heights[c];
    el("rect", { x: k * s, y: r * s, width: s - 1, height: s - 1, rx: 1.5, fill: h === 4 ? css("--dome") : shade[h], stroke: h ? css("--stone-edge") : "none", "stroke-width": .6 }, g);
    if (h > 0 && h < 4 && s >= 13) txt(g, k * s + 2, r * s + 7, h, { "font-size": 6, fill: "#6b7468" });
  }
  for (const p of [0, 1]) for (const c of n.workers[p])
    el("circle", { cx: (c % 5) * s + s / 2 - .5, cy: Math.floor(c / 5) * s + s / 2 - .5, r: s * 0.32, fill: playerColor(p), stroke: "#fff", "stroke-width": .8 }, g);
  if (n.move) {
    const [f, t, b] = n.move, col = playerColor(n.moved_by);
    el("line", { x1: (f % 5) * s + s / 2, y1: Math.floor(f / 5) * s + s / 2, x2: (t % 5) * s + s / 2, y2: Math.floor(t / 5) * s + s / 2, stroke: col, "stroke-width": 1.6, opacity: .9 }, g);
    if (b >= 0) el("rect", { x: (b % 5) * s + 1, y: Math.floor(b / 5) * s + 1, width: s - 3, height: s - 3, rx: 1.5, fill: "none", stroke: col, "stroke-width": 1.6 }, g);
  }
}

function crumb(tree) {
  const c = byId("crumb");
  if (!tree || !tree.nodes.length) { c.textContent = ""; return; }
  const r = tree.nodes[0];
  const shownCount = tree.nodes.length;
  c.innerHTML = (treeParams.root ? `Exploring from #${r.id}${r.move ? ` (${NAMES[r.moved_by]} ${r.move_text})` : ""}<button id="crumbTop">Back to the top</button> ` : "") +
    `${shownCount} of ${tree.total_nodes.toLocaleString()} nodes shown, ${tree.simulations.toLocaleString()} simulations`;
  const b = byId("crumbTop");
  if (b) b.onclick = () => { treeParams.root = 0; refreshTree(); };
}

function stepExplain() {
  const s = stepInfo;
  const label = { selected: "1. Select", evaluated: "2. Expand and evaluate", backed_up: "3. Back up", done: "Finished" }[s.phase] || "";
  const body = s.explain.replace(/^[^:]+: /, "");
  return `<b>${label}.</b> ${body.charAt(0).toUpperCase() + body.slice(1)}`;
}

// pan and zoom
function applyView() { byId("treeG").setAttribute("transform", `translate(${tv.x},${tv.y}) scale(${tv.k})`); }
function fitTree() {
  if (!treeLayout) return;
  const box = byId("tree").getBoundingClientRect();
  const k = Math.min(1.4, (box.width - 30) / treeLayout.width, (box.height - 30) / treeLayout.height);
  tv.k = Math.max(0.05, k);
  tv.x = (box.width - treeLayout.width * tv.k) / 2;
  tv.y = 16;
  applyView();
}
function readableView() {   // fit if that keeps nodes legible, otherwise zoom to a readable size centred on the root
  if (!treeLayout) return;
  const boxEl = byId("treebox");
  if (!document.fullscreenElement && !boxEl.style.height)
    boxEl.style.minHeight = "";
  const box = byId("tree").getBoundingClientRect();
  const fit = Math.min(1.2, (box.width - 30) / treeLayout.width, (box.height - 30) / treeLayout.height);
  if (fit >= 0.7) return fitTree();
  tv.k = 0.75;
  tv.x = box.width / 2 - treeLayout.rootX * tv.k;
  tv.y = 16;
  applyView();
}
function zoomAt(f, px, py) {
  const box = byId("tree").getBoundingClientRect();
  if (px === undefined) { px = box.width / 2; py = box.height / 2; }
  const k2 = Math.min(4, Math.max(0.05, tv.k * f));
  tv.x = px - (px - tv.x) * k2 / tv.k; tv.y = py - (py - tv.y) * k2 / tv.k; tv.k = k2;
  applyView();
}
(function panZoom() {
  const svg = byId("tree");
  let drag = null;
  svg.addEventListener("wheel", e => { e.preventDefault(); const b = svg.getBoundingClientRect(); zoomAt(e.deltaY < 0 ? 1.15 : 1 / 1.15, e.clientX - b.left, e.clientY - b.top); }, { passive: false });
  svg.addEventListener("pointerdown", e => { drag = { x: e.clientX, y: e.clientY, tx: tv.x, ty: tv.y }; tv.moved = false; });
  window.addEventListener("pointermove", e => {
    if (!drag) return;
    const dx = e.clientX - drag.x, dy = e.clientY - drag.y;
    if (Math.abs(dx) + Math.abs(dy) > 4) { tv.moved = true; svg.classList.add("dragging"); }
    tv.x = drag.tx + dx; tv.y = drag.ty + dy; applyView();
  });
  window.addEventListener("pointerup", () => { drag = null; svg.classList.remove("dragging"); setTimeout(() => tv.moved = false, 0); });
  byId("zIn").onclick = () => zoomAt(1.25);
  byId("zOut").onclick = () => zoomAt(0.8);
  byId("zFit").onclick = fitTree;
  byId("zFull").onclick = () => {
    const box = byId("treebox");
    if (document.fullscreenElement) document.exitFullscreen();
    else if (box.requestFullscreen) box.requestFullscreen().catch(() => { box.style.height = box.style.height ? "" : "85vh"; setTimeout(fitTree, 50); });
    else { box.style.height = box.style.height ? "" : "85vh"; setTimeout(fitTree, 50); }
  };
  document.addEventListener("fullscreenchange", () => setTimeout(fitTree, 80));
})();

async function refreshTree() {
  const target = view === null ? "live" : view;
  const r = await api("tree", treeReq(target));
  if (view === null) liveTree = r.tree; else past.tree = r.tree;
  drawTree();
}

// ---------- value chart, log, status, timeline ---------------------------------------------------------
function drawEval() {
  const svg = byId("eval");
  svg.innerHTML = "";
  const W = 520, H = 170, L = 34, R = 8, T = 10, B = 22;
  const n = G ? Math.max(G.log.length, 6) : 6;
  const X = e => L + (W - L - R) * (e + 0.5) / n, Y = b => T + (H - T - B) * (1 - b);
  for (const [b, lab] of [[1, "100%"], [0.75, "75%"], [0.5, "50%"], [0.25, "25%"], [0, "0%"]]) {
    el("line", { x1: L, x2: W - R, y1: Y(b), y2: Y(b), stroke: css("--line"), "stroke-dasharray": b === 0.5 ? "4 3" : "none" }, svg);
    txt(svg, L - 5, Y(b) + 3, lab, { "text-anchor": "end", "font-size": 9.5, fill: css("--muted") });
  }
  txt(svg, L + 4, Y(1) + 11, "Blue ahead", { "font-size": 10, fill: css("--blue") });
  txt(svg, L + 4, Y(0) - 4, "Gold ahead", { "font-size": 10, fill: css("--gold") });
  txt(svg, W - R, H - 5, "move", { "text-anchor": "end", "font-size": 9.5, fill: css("--muted") });
  if (view !== null) el("line", { x1: X(view), x2: X(view), y1: T, y2: H - B, stroke: css("--ink"), "stroke-width": 1.5, opacity: .5 }, svg);
  if (!G || !G.evals.length) {
    txt(svg, (L + W) / 2, H / 2 - 8, "Fills in as the agent thinks.", { "text-anchor": "middle", "font-size": 12, fill: css("--muted") });
    return;
  }
  const pts = G.evals.slice().sort((a, b) => a.entry - b.entry).map(e => [X(e.entry), Y(e.blue), e]);
  const defs = el("defs", {}, svg);
  const cpA = el("clipPath", { id: "above" }, defs); el("rect", { x: 0, y: 0, width: W, height: Y(0.5) }, cpA);
  const cpB = el("clipPath", { id: "below" }, defs); el("rect", { x: 0, y: Y(0.5), width: W, height: H }, cpB);
  const area = `M${pts[0][0]},${Y(0.5)} ` + pts.map(p => `L${p[0]},${p[1]}`).join(" ") + ` L${pts[pts.length - 1][0]},${Y(0.5)} Z`;
  el("path", { d: area, fill: css("--blue"), opacity: .35, "clip-path": "url(#above)" }, svg);
  el("path", { d: area, fill: css("--gold"), opacity: .45, "clip-path": "url(#below)" }, svg);
  el("polyline", { points: pts.map(p => p[0].toFixed(1) + "," + p[1].toFixed(1)).join(" "), fill: "none", stroke: css("--ink"), "stroke-width": 1.8 }, svg);
  for (const [x, y, e] of pts) {
    const c = el("circle", { cx: x, cy: y, r: e.proven ? 4.5 : 3.5, fill: e.proven ? css("--ink") : css("--panel"), stroke: css("--ink"), "stroke-width": 1.5, style: "cursor:pointer" }, svg);
    const t = el("title", {}, c); t.textContent = `Move ${e.entry + 1}: Blue ${Math.round(100 * e.blue)}%${e.proven ? " (proven)" : ""}${e.by === "hint" ? ", from a hint" : ""}`;
    c.addEventListener("click", () => setView(e.entry));
  }
}

function drawLog() {
  const ul = byId("log");
  ul.innerHTML = "";
  if (!G) return;
  G.log.forEach((e, i) => {
    const li = document.createElement("li");
    if (view === i) li.className = "cur";
    li.innerHTML = `<span class="n">${i + 1}</span><span class="dot" style="background:${playerColor(e.player)}"></span>
      <span>${NAMES[e.player]}${e.player === G.human ? " (you)" : ""} ${e.text}</span><span class="note">${e.note || ""}</span>`;
    li.addEventListener("click", () => setView(i));
    ul.appendChild(li);
  });
  if (view === null) ul.scrollTop = ul.scrollHeight;
}

function paused() { return byId("pause").checked && G && G.turn === G.agent && G.phase === "play" && view === null; }

function drawStatus() {
  const s = byId("status");
  if (!G) return;
  const you = G.human, them = G.agent, span = p => `<span class="who-${p}">${NAMES[p]}</span>`;
  let msg;
  if (view !== null) {
    const e = G.log[view];
    msg = `Reviewing move ${view + 1}: ${span(e.player)} ${e.text}. The board shows the position before it.`;
  } else if (G.phase === "over") msg = `${span(G.winner)} wins: ${G.reason}. ${G.winner === you ? "Well played." : "Take back a move, review the game, or start a new one."}`;
  else if (G.turn === them && paused() && !busy) msg = `${span(them)} is paused. Step through its search below, then press Let the agent move.`;
  else if (G.turn === them) msg = busy ? `${span(them)} is thinking…` : `${span(them)} to move.`;
  else if (G.phase === "place") msg = `You are ${span(you)}. Place worker ${G.workers[you].length + 1} of 2 on any empty cell.`;
  else if (!sel) msg = `Your move as ${span(you)}: pick one of your workers.`;
  else if (sel.to === undefined) msg = `Move the worker on ${cellName(sel.from)} to a marked cell (a green ring wins at once). Click it again to cancel.`;
  else msg = `Now build next to ${cellName(sel.to)} on a dashed square.`;
  s.innerHTML = msg;
  byId("net").textContent = net.source ? `Network: ${net.source}` + (net.iteration ? `, iteration ${net.iteration}` : "") +
    (net.blocks ? `, ${net.blocks}×${net.channels} ResNet` : "") + (net.device ? ` on ${net.device}` : "") : "";
  byId("undo").disabled = busy || view !== null || !G.log.some(e => e.player === G.human);
  byId("letMove").hidden = !paused();
  const tl = byId("tl");
  tl.max = G.log.length; tl.value = view === null ? G.log.length : view;
  byId("tlLabel").innerHTML = view === null ? `<b>Now</b>, after ${G.log.length} move${G.log.length === 1 ? "" : "s"}. Drag the slider to review earlier moves and the thinking behind them.` :
    `<b>Move ${view + 1} of ${G.log.length}</b>: ${NAMES[G.log[view].player]} ${G.log[view].text}${G.log[view].thought !== undefined ? "" : " (no search recorded)"}`;
  byId("tlPrev").disabled = (view === null ? G.log.length : view) <= 0;
  byId("tlNext").disabled = view === null;
  byId("toNow").hidden = view === null;
  const f = view !== null ? G.frames[view] : null;
  byId("playHere").hidden = !(f && f.turn === G.human && f.phase !== "over");
}

function render() {
  drawBoard(); drawThinking(); drawTree(); drawEval(); drawLog(); drawStatus();
  if (window.google && google.colab && google.colab.output && google.colab.output.setIframeHeight)
    google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
}

async function setView(k) {
  if (!G) return;
  hover = null; previewNode = null; sel = null;
  if (k === null || k >= G.log.length) { view = null; treeParams.root = 0; render(); return; }
  view = Math.max(0, k);
  treeParams.root = 0;
  past = { thought: null, placement: null, tree: null };
  render();
  const want = view;
  const r = await api("thought", { entry: view, tree: treeReq(view) });
  if (view !== want) return;
  past = { thought: r.thought, placement: r.placement, tree: r.tree || null };
  render();
}

// ---------- flow ---------------------------------------------------------------------------------
async function onCell(c) {
  if (!humanToMove()) return;
  if (G.phase === "place") { hintToken++; await api("place", { cell: c }); render(); return next(); }
  if (!sel) { if (G.workers[G.human].includes(c) && legalFrom(c).length) { sel = { from: c }; hover = null; } return render(); }
  if (sel.to === undefined) {
    if (c === sel.from) { sel = null; return render(); }
    if (G.workers[G.human].includes(c) && legalFrom(c).length) { sel = { from: c }; return render(); }
    const ms = legalFrom(sel.from).filter(m => m[1] === c);
    if (!ms.length) return;
    if (ms[0][2] < 0) return submit(sel.from, c, -1);
    sel = { from: sel.from, to: c }; return render();
  }
  if (!legalFrom(sel.from).some(m => m[1] === sel.to && m[2] === c)) { sel = null; return render(); }
  return submit(sel.from, sel.to, c);
}

async function submit(f, t, b) {
  hintToken++;
  sel = null; liveThought = null; liveTree = null; stepInfo = null; treeParams.root = 0;
  await api("move", { frm: f, to: t, build: b });
  render();
  next();
}

const sleep = ms => new Promise(r => setTimeout(r, ms));

async function thinkLoop(token, isAgent) {
  const target = +byId("sims").value;
  for (;;) {
    if (isAgent ? token !== agentToken : token !== hintToken) return false;
    const done = liveThought && G && liveThought.player === G.turn ? liveThought.simulations : 0;
    const ask = G && G.phase === "play" ? Math.max(1, Math.min(chunk, target - done)) : chunk;
    const t0 = performance.now();
    const r = await api("think", { sims: ask, tree: G.phase === "play" ? treeReq("live") : null });
    const dt = performance.now() - t0;
    if (isAgent ? token !== agentToken : token !== hintToken) return false;
    if ("thought" in r) liveThought = r.thought;
    if ("placement" in r) livePlacement = r.placement;
    if ("tree" in r) liveTree = r.tree;
    if (ask === chunk) chunk = Math.max(8, Math.min(512, Math.round(chunk * (dt > 0 ? 350 / dt : 2))));
    render();
    if (G.phase === "place") { if (livePlacement && livePlacement.progress >= 1) return true; continue; }
    if (!r.thought || r.thought.done || r.thought.simulations >= target) return true;
  }
}

async function next() {
  if (!G || G.phase === "over") { render(); return; }
  if (G.turn === G.agent) {
    if (paused()) { busy = false; render(); return; }
    busy = true; render();
    const token = ++agentToken;
    const finished = await thinkLoop(token, true);
    if (!finished || token !== agentToken) { busy = false; render(); return; }
    await sleep(250);
    const played = await api("agent_play");
    if (token !== agentToken) return;
    busy = false; livePlacement = null; liveThought = null; liveTree = null; stepInfo = null; treeParams.root = 0;
    render();
    return next();
  }
  if (byId("hints").checked && G.phase === "play") await thinkLoop(++hintToken, false);
}

let stepping = false;
async function step(action, n) {
  if (view !== null || !G || G.phase !== "play" || stepping) return;
  hintToken++;
  stepping = true;
  if (!liveThought || liveThought.player !== G.turn) byId("explain").innerHTML = "Starting a search for this position (the exact solver checks it first)…";
  ["sSelect", "sExpand", "sBackup", "sOne", "sMany"].forEach(id => byId(id).disabled = true);
  let r;
  try { r = await api("step", { action, n: n || 1, tree: treeReq("live") }); } finally { stepping = false; }
  if ("thought" in r) liveThought = r.thought;
  if ("tree" in r) liveTree = r.tree;
  render();
}

async function newGame(human) {
  agentToken++; hintToken++;
  Object.assign(treeParams, { root: 0 });
  sel = null; hover = null; liveThought = null; livePlacement = null; liveTree = null; busy = false; view = null; stepInfo = null;
  await api("new", { human, sims: +byId("sims").value, solver: byId("solver").checked });
  render(); next();
}

byId("newBlue").onclick = () => newGame(0);
byId("newGold").onclick = () => newGame(1);
byId("undo").onclick = async () => { agentToken++; hintToken++; busy = false; sel = null; liveThought = null; liveTree = null; livePlacement = null; stepInfo = null; await api("undo"); render(); next(); };
byId("playHere").onclick = async () => { const k = view; agentToken++; hintToken++; busy = false; view = null; liveThought = null; liveTree = null; stepInfo = null; await api("rewind", { entry: k }); render(); next(); };
byId("toNow").onclick = () => setView(null);
byId("tl").oninput = e => setView(+e.target.value);
byId("tlPrev").onclick = () => setView((view === null ? G.log.length : view) - 1);
byId("tlNext").onclick = () => setView(view === null ? null : view + 1);
byId("sims").onchange = () => api("settings", { sims: +byId("sims").value });
byId("solver").onchange = () => api("settings", { solver: byId("solver").checked });
byId("hints").onchange = () => { if (!byId("hints").checked) { hintToken++; if (liveThought && G && liveThought.player === G.human) { liveThought = null; liveTree = null; } render(); } else next(); };
byId("thoughts").onchange = render;
byId("pause").onchange = () => { if (!byId("pause").checked) next(); else { agentToken++; busy = false; render(); } };
byId("letMove").onclick = () => { byId("pause").checked = false; next(); };
byId("sSelect").onclick = () => step("select");
byId("sExpand").onclick = () => step("expand");
byId("sBackup").onclick = () => step("backup");
byId("sOne").onclick = () => step("one");
byId("sMany").onclick = () => step("many", 25);
for (const [id, key, fmt] of [["tMax", "max_nodes", v => v], ["tPrune", "prune", v => v + "%"], ["tDepth", "depth", v => v]]) {
  byId(id).oninput = e => { byId(id + "V").textContent = fmt(e.target.value); };
  byId(id).onchange = e => { treeParams[key] = +e.target.value; refreshTree(); };
}
byId("tHide").onchange = e => { treeParams.unvisited = !e.target.checked; refreshTree(); };
for (const id of ["tBoards", "tPuct", "tMinimal"]) byId(id).onchange = drawTree;
document.addEventListener("keydown", e => { if (e.key === "Escape" && sel) { sel = null; render(); } });
window.addEventListener("resize", () => { if (treeLayout) readableView(); });

(async () => {
  try {
    const r = await api("state", { tree: treeReq("live") });
    if (r.settings) { byId("sims").value = String(r.settings.sims); byId("solver").checked = r.settings.solver; }
    if (r.thought) liveThought = r.thought;
    if (r.tree) liveTree = r.tree;
    if (!G.log.length) await api("new", { human: 0, sims: +byId("sims").value, solver: byId("solver").checked });
    render(); next();
  } catch (e) {
    byId("status").textContent = "Can't reach the agent. Run the cell that starts it, then reload this widget.";
    byId("err").textContent = String(e);
  }
})();
</script>
</body>
</html>

Overwriting play.html


In [20]:
import pybind11
ext = sysconfig.get_config_var("EXT_SUFFIX")
cmd = (f"c++ -O3 -march=native -std=c++17 -shared -fPIC -pthread -I{pybind11.get_include()} "
       f"-I{sysconfig.get_paths()['include']} santorini_py.cpp -o santorini{ext}")
log("building the C++ module (~20 s) ...")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if r.returncode:
    print(r.stdout, r.stderr); raise RuntimeError("build failed")
sys.path.insert(0, os.getcwd())
import importlib, santorini, santorini_agent
importlib.reload(santorini_agent)
log("module built")

[15:55:25 |   0.1 min] building the C++ module (~20 s) ...
[15:55:56 |   0.6 min] module built


## Load the network
- `"drive"` (default) reads the checkpoint the training notebooks keep on Google Drive.
- `"upload"` asks you for a `latest.pkl` or `weights_XXXX.msgpack` file.
- `"none"` plays with search and the solver only, without a network.

In [21]:
SOURCE = "drive"
DRIVE_FILE = "/content/drive/MyDrive/santorini_az/latest.pkl"

from santorini_agent import Net, Session, show
ckpt = None
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ckpt = DRIVE_FILE
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"{ckpt} not found. Set DRIVE_FILE, or SOURCE = 'upload'.")
elif SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    ckpt = next(iter(up))
t = time.time()
net = Net(ckpt)
log(f"network loaded in {time.time() - t:.1f}s: {net.info}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[15:55:57 |   0.6 min] network loaded in 0.3s: {'source': 'latest.pkl', 'blocks': 10, 'channels': 128, 'params': 3152026, 'device': 'cpu', 'iteration': 13, 'trained_on': 'gpu', 'test_won_top1_winning': 0.516}


## Play
Re-running this cell reopens the board and keeps the current game. Use the buttons under the board to start a new game as Blue (you move first) or Gold.

In [22]:
if "session" not in globals() or session.net is not net:
    session = Session(net)
show(session)

**Playing outside Colab:** with the same files in one folder and the module built, run `python santorini_agent.py --ckpt latest.pkl` and open `http://127.0.0.1:8000`.